# 초구 예측 — model_16 (3단계: 위치 상황 보정 월 단위 재학습)

- model_14: 위치 상황 보정 채택 (README 4-29), model_15 타자 성향 채택 안 함 (4-30)
- 0장: model_14 위에서 남은 변수 묶음 다시 확인 (0-1 ~ 0-8, README 4-31 ~ 4-36) -> 모두 남는 몫 없음, 위치 변수 단계 끝
- 0-9 재학습 주기: 상황 보정을 월 단위로 -> 2019 +1.67 (95% +0.82 ~ +2.70) · 2020 +0.29 (−0.30 ~ +0.94) ×1000
- 모델: model_14에서 **상황 보정 LightGBM의 재학습 주기만** 시즌 단위 -> 월 단위로 (사용자 결정, 구종 model_09과 같은 방식)
  - 달마다 그 달 첫 경기 전날까지 모든 초구(2018부터, 지난 시즌 + 그 시즌 지난달까지)로 다시 학습, 3월은 4월에·10월은 9월에 합침
  - 변수·파라미터(잎 7 · 잎당 최소 20 · λ 1000)·트리 수(168)·위치 정규분포·구종은 model_14 그대로
- 비교 대상: model_14

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys
if not os.path.exists("fp_common.py"):  # model/ 폴더에서 열면 한 단계 위(데이터·fp_common·results 있는 곳)로
    os.chdir("..")
sys.path.insert(0, os.getcwd())  # fp_common을 찾도록
import numpy as np
import pandas as pd
import fp_common as fc

MODEL = "model_16"

m, names = fc.load_data()
fp, y_type, y_zone, val, test = fc.first_pitches(m)
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]
print(f"초구 {len(fp):,}개 (검증 {val.sum():,} / 평가 {test.sum():,})")
import lightgbm as lgb
from scipy.special import log_softmax
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)

초구 50,709개 (검증 13,458 / 평가 12,615)


## 0. 모델 전 확인

### 0-1. model_14 위 포수·구장·월

- 구조: model_14과 같음 (model_13 위치 log 확률을 출발점으로 24칸 LightGBM, 학습 2018 -> 2019, 파라미터 잎 7·잎당 최소 20·λ 1000)
- 변수 (4-28와 같음, `fc.group_residual_loc`: 전날까지 누적표라 미래 정보 없음)
  - `catcher_H·S·C` 포수별 H/S/C 잔차 (그 포수와 던질 때 위치 확률이 빗나간 정도, 전날까지 / (공 수 + 50))
  - `venue_H·S·C` 구장별 H/S/C 잔차 (같은 방식), `month` 월 (3월은 4월, 10월은 9월에)
  - 잔차의 기준 확률은 model_13 위치 (상황 보정 전, 4-28와 같음) -> 트리가 상황 변수와 함께 씀
- 비교 두 가지 (이득 ×1000 = model_14 구조 − 후보, 2019 24칸 log loss, 경기 단위 95% 구간)
  - **트리 수 고정 168** (model_14과 같은 설정, 기준 = model_14 그대로 재현)
  - **트리 수 조기 종료** (2018 경기 20%로, 0-3 방식) -> 변수가 늘면 트리 수가 달라질 수 있어서
- 묶음 전체 + 하나씩 (포수 / 구장 / 월)

In [2]:
DIAG = "model_14"
pp18 = fc.load_results([DIAG])[DIAG]
LOC_REF, LOC_REF_HL = "decay", 173.0  # model_13 위치 정규분포 (model_14과 같음)
LOC_HL, K_TYPE, K_HAND = 35.4, 1000.0, 30.0
LOC_SEASON_F = 0.707
LOC_AC = (None, 12.5, 17.3)
LOC_VAR = (200.0, 9.74, False, 316.0)
ZONE_M = 32
LOC_SIT = ["base1", "base2", "base3", "first_open", "sdiff", "inning", "out_cn", "home", "batorder"]  # model_14 상황 변수
LOC_FEAT18 = CLASSES + ["same_side"] + LOC_SIT
LOC_LGB18, LOC_ROUNDS18 = {"num_leaves": 7, "min_data_in_leaf": 20, "lambda_l2": 1000.0}, 168  # model_14 LightGBM
# 24칸 적분 함수: fc.zone_from_gauss_cov(g, q, m=32) (model_03부터)
def zone_probs(g, q, cov=True):  # 24칸 확률: cov=True 공분산 반영 (model_03) / False 독립 가정 (model_02까지)
    return fc.zone_from_gauss_cov(g, q, ZONE_M) if cov else fc.zone_from_gauss(g, q)

L_COLS = MOM + ZONES  # 당길 열: 위치 모멘트 5개(x, 높이 비율, 제곱, 곱) + 24칸 원-핫
K1, K2 = ["pitcher_code", "pitch_type"], ["pitcher_code", "pitch_type", "same_side"]  # 투수×구종 / 투수×구종×손
KR = ["pitch_type", "same_side"]  # 기준점 그룹: 분석 대상 투수 구종×손

def loc_league(q, year, ref="all", ref_hl=None):  # 기준점 (n, 29): "all" 지난 시즌 전부 / "last" 직전 시즌만 / "decay" 전날까지 반감기 ref_hl(공 수) 가중
    past = fp[fp["GYEAR"] < year]  # 지난 시즌까지
    overall = past[L_COLS].mean().to_numpy()  # 그 구종×손 기록이 없을 때 쓰는 전체 평균 (지난 시즌까지)
    if ref == "decay":
        S, N = fc.lookup(fc.prior_table(fp, KR, L_COLS, ref_hl), q, KR, L_COLS)  # 구종×손별 전날까지 가중 합계, 개수
        return np.where(N > 0, S / np.maximum(N, 1e-12), overall)  # 가중 평균, 기록 없으면 전체
    if ref == "last":
        past = fp[fp["GYEAR"] == year - 1]  # 직전 시즌만
    lg = past.groupby(KR)[L_COLS].mean()
    pr = lg.reindex(pd.MultiIndex.from_frame(q[KR]))  # 공마다 그 구종×손 행
    return pr.fillna(pd.Series(overall, index=L_COLS)).to_numpy()

def loc_sn(q, half_life, season_f, cache=True):  # 투수×구종, 투수×구종×손 전날까지 초구 위치 합계 S·개수 N 두 쌍 (시즌 초 배수 반영)
    return [fc.lookup(fc.prior_table(fp, keys, L_COLS, half_life, season_f=season_f, cache=cache), q, keys, L_COLS, season_f=season_f)
            for keys in (K1, K2)]

def shift_mean(M, dx, dz):  # 모멘트 M (n, 29)의 평균만 (dx, dz) 옮김, 분산·공분산 그대로 (24칸 원-핫 열은 정규분포에 안 쓰여 그대로)
    M = M.copy()
    mx, mz = M[:, 0], M[:, 1]
    nx, nz = mx + dx, mz + dz
    M[:, 2] += nx ** 2 - mx ** 2  # E[x²] = 분산 + 평균²
    M[:, 3] += nz ** 2 - mz ** 2
    M[:, 4] += nx * nz - mx * mz  # E[x·z] = 공분산 + 평균 곱
    M[:, 0], M[:, 1] = nx, nz
    return M

# 0-0 아닌 공 (구종 의심 제외): 모든 카운트 위치 성향 재료
MO = m[~m["label_doubt"] & ~(m["ball_before"].eq(0) & m["strike_before"].eq(0))].copy()
MO["x"], MO["zn"] = MO["plate_x_in_CM"], MO["z_norm"]  # 좌우 cm (타자 기준), 높이 비율 (존 아랫선 0, 윗선 1)
MO["cnt"] = MO["ball_before"].astype(int).astype(str) + "-" + MO["strike_before"].astype(int).astype(str)  # 카운트
CK = ["cnt", "pitch_type", "same_side"]  # 카운트 효과 그룹
_MO_RES = {}

def mo_resid(ref_hl):  # 0-0 아닌 공마다 위치 − 분석 대상 투수 그 카운트×구종×손 전날까지 평균 (반감 ref_hl구) -> 잔차 표 (rx 좌우 cm, rz 높이 비율)
    if ref_hl not in _MO_RES:
        S, N = fc.lookup(fc.prior_table(MO, CK, ["x", "zn"], ref_hl, cache=False), MO, CK, ["x", "zn"])
        ref = np.where(N > 0, S / np.maximum(N, 1e-12), np.nan)  # 그 카운트×구종×손 기록이 없으면 빈 값 (셈에서 빠짐)
        _MO_RES[ref_hl] = MO[K2 + ["game_date", "gameID", "pit_no"]].assign(rx=MO["x"].to_numpy() - ref[:, 0], rz=MO["zn"].to_numpy() - ref[:, 1])
    return _MO_RES[ref_hl]

def ac_sn(q, ref_hl, hl_b, season_f, cache=True):  # 공마다 투수×구종×손 다른 카운트 잔차 전날까지 합 S·개수 N (n, 2)
    d = mo_resid(ref_hl)
    return fc.lookup(fc.prior_table(d, K2, ["rx", "rz"], hl_b, season_f=season_f, cache=cache), q, K2, ["rx", "rz"], season_f=season_f)

MCOLS = ["x", "zn", "xx", "zz", "xz"]  # 위치 모멘트 (좌우, 높이 비율, 제곱, 곱)
_MO_MOM, _WT = {}, {}

def mo_mom(ref_hl):  # 다른 카운트 잔차 모멘트 표 (x = 좌우 잔차, zn = 높이 비율 잔차, 제곱·곱)
    if ref_hl not in _MO_MOM:
        d = mo_resid(ref_hl)
        _MO_MOM[ref_hl] = d.assign(xx=d["rx"] ** 2, zz=d["rz"] ** 2, xz=d["rx"] * d["rz"]).rename(columns={"rx": "x", "rz": "zn"})
    return _MO_MOM[ref_hl]

def within_tab(name, data, year):  # 구종×손별 투수 안 분산·공분산 (지난 시즌까지 투수×구종×손 분산의 (공 수 − 1) 가중 평균) 표와 전체 값
    if (name, year) not in _WT:
        past = data[data["game_date"].dt.year < year].dropna(subset=["x", "zn"])
        g = past.groupby(K2)
        n = g.size()
        w = (n - 1).clip(lower=0)
        c = pd.DataFrame({"vx": g["x"].var(), "vz": g["zn"].var(),
                          "cxz": (g["xz"].mean() - g["x"].mean() * g["zn"].mean()) * n / (n - 1).clip(lower=1)}).fillna(0)  # 표본 분산·공분산
        num = c.mul(w, axis=0).groupby(level=["pitch_type", "same_side"]).sum()
        wg = w.groupby(level=["pitch_type", "same_side"]).sum()  # 구종×손별 자유도 합 (표본 크기)
        _WT[(name, year)] = (num.div(wg, axis=0), c.mul(w, axis=0).sum() / w.sum(), wg)
    return _WT[(name, year)]

def v0_of(q, name, data, year, k0=0.0):  # 공마다 그 구종×손 투수 안 분산·공분산 (n, 3): 자유도 합이 적으면 전체 구종 값 쪽으로 k0만큼 당김
    tab, allv, wg = within_tab(name, data, year)
    idx = pd.MultiIndex.from_frame(q[KR])
    a = allv.to_numpy()[None, :]  # 전체 구종 투수 안 분산
    v = tab.reindex(idx).to_numpy()
    n = wg.reindex(idx).fillna(0).to_numpy()[:, None]  # 그 구종×손 자유도 합
    v = np.where(np.isnan(v), a, v)
    return np.where(n + k0 > 0, (n * v + k0 * a) / np.maximum(n + k0, 1e-12), a)  # 예: 너클 초구 3구(자유도 2)면 거의 전체 값

def own_var(q, year, ref_hl, hl_var, season_f, k0=0.0, cache=True):  # 공마다 다른 카운트 퍼짐 -> 초구 단위로 바꾼 자기 분산·공분산 (n, 3), (가중) 공 수 (n,), V0 (n, 3)
    data = mo_mom(ref_hl)
    S, N = fc.lookup(fc.prior_table(data, K2, MCOLS, hl_var, season_f=season_f, cache=cache), q, K2, MCOLS, season_f=season_f)
    e = S / np.maximum(N, 1e-12)
    v = np.column_stack([e[:, 2] - e[:, 0] ** 2, e[:, 3] - e[:, 1] ** 2, e[:, 4] - e[:, 0] * e[:, 1]])  # 자기 잔차 분산·공분산
    V0 = v0_of(q, "초구", fp, year, k0)
    V0m = v0_of(q, ("다른 카운트", ref_hl), data, year, k0)
    r = np.sqrt(np.maximum(v[:, :2], 0) / V0m[:, :2])  # 좌우·높이 SD 비율 (자기 / 투수 안)
    v = np.column_stack([V0[:, 0] * r[:, 0] ** 2, V0[:, 1] * r[:, 1] ** 2, v[:, 2] / np.sqrt(V0m[:, 0] * V0m[:, 1]) * np.sqrt(V0[:, 0] * V0[:, 1])])
    n = N[:, 0]
    return np.where(n[:, None] > 0, np.nan_to_num(v), 0.0), n, V0

def apply_var(M, v, n, V0, k_var, unc, N2, k_hand):  # 모멘트 M의 분산·공분산을 따로 당긴 값으로 바꿈 (평균 그대로)
    V = (n[:, None] * v + k_var * V0) / (n[:, None] + k_var)  # 투수 안 분산 쪽으로 k_var만큼
    if unc:
        V = V * (1 + 1 / (N2[:, :1] + k_hand))  # 평균을 덜 알수록 넓힘
    M = M.copy()
    mx, mz = M[:, 0], M[:, 1]
    M[:, 2], M[:, 3], M[:, 4] = V[:, 0] + mx ** 2, V[:, 1] + mz ** 2, V[:, 2] + mx * mz  # 분산 -> 모멘트
    return M

def loc_online(q, year, half_life, k_type, k_hand, ref="all", ref_hl=None, season_f=1.0, ac=None, var=None):  # 위치 모멘트 (n, 29)
    P = loc_league(q, year, ref, ref_hl)  # 출발: 기준점
    if ac is not None:  # 모든 카운트 위치 성향 (model_12)
        S, N = ac_sn(q, ac[0], ac[1], season_f)
        d = S / (N + ac[2])
        P = shift_mean(P, d[:, 0], d[:, 1])
    (S1, N1), (S2, N2) = loc_sn(q, half_life, season_f)
    M = fc.shrink(S2, N2, fc.shrink(S1, N1, P, k_type), k_hand)  # 투수×구종 -> 투수×구종×손
    if var is not None:  # 분산 따로 (model_13): var = (hl_var, k_var, unc, k0)
        v, n, V0 = own_var(q, year, None if ac is None else ac[0], var[0], season_f, var[3])
        M = apply_var(M, v, n, V0, var[1], var[2], N2, k_hand)
    return M

def loc_given_type(q, year, t, half_life, k_type, k_hand, ref="all", ref_hl=None, season_f=1.0, ac=None, var=None, cov=True):  # 구종이 t일 때 24칸 확률 (n, 24)과 좌표 정규분포
    qt = fc.as_type(q, t)
    g = fc.gauss(loc_online(qt, year, half_life, k_type, k_hand, ref, ref_hl, season_f, ac, var), qt)
    return zone_probs(g, qt, cov), g



LOC17 = dict(half_life=LOC_HL, k_type=K_TYPE, k_hand=K_HAND, ref=LOC_REF, ref_hl=LOC_REF_HL, season_f=LOC_SEASON_F, ac=LOC_AC, var=LOC_VAR)
PZ_TRUE = np.full((len(fp), len(ZONES)), np.nan)  # 실제 구종 기준 model_13 24칸 확률
for yr in (2018, fc.VAL_YEAR):
    k = fp["GYEAR"].eq(yr).to_numpy()
    PZ_TRUE[k] = loc_given_type(fp[k], yr, fp.loc[k, "pitch_type"].to_numpy(), **LOC17)[0]

from scipy.special import log_softmax
SIT_L = fc.situation_features(m, fp)
P3 = PZ_TRUE @ fc.GROUP2  # model_13 H/S/C 확률 (2017·2020은 빈 값 -> 잔차에서 안 셈)
RES = pd.concat([fc.group_residual_loc(fp, P3, ["catcher_code"], fp, prefix="catcher"),
                 fc.group_residual_loc(fp, P3, ["VENUE_ID"], fp, prefix="venue")], axis=1)
XLOC = pd.concat([fp[CLASSES + ["same_side"]], SIT_L, fc.opponent_features(m, fp)[["home"]], RES], axis=1)
XLOC["month"] = fp["game_date"].dt.month.clip(4, 9)
XLOC["first_open"] = (((SIT_L["base2"] + SIT_L["base3"]) > 0) & SIT_L["base1"].eq(0)).astype(int)
hsc = lambda pre: [f"{pre}_{c}" for c in fc.HSC]
GROUPS = {"포수·구장·월 전부": hsc("catcher") + hsc("venue") + ["month"], "포수": hsc("catcher"), "구장": hsc("venue"), "월": ["month"]}
NZ = len(ZONES)
INIT_L = np.log(np.clip(PZ_TRUE, 1e-6, None))
INIT_L -= INIT_L.mean(1, keepdims=True)
yr_l = fp["GYEAR"].to_numpy()
BASE_L = dict(objective="multiclass", num_class=NZ, learning_rate=0.05, feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1,
              verbose=-1, seed=0, deterministic=True, force_row_wise=True, num_threads=4)  # model_14과 같음
itr = np.flatnonzero(yr_l == 2018)
i19 = np.flatnonzero(val)
gid_all = fp["gameID"].to_numpy()
g18 = np.unique(gid_all[itr])
es = np.isin(gid_all[itr], np.random.default_rng(1).choice(g18, len(g18) // 5, replace=False))  # 조기 종료용 경기 20%

def fit_pred(cols, rounds=None):  # cols로 2018 학습 -> 2019 24칸 확률, 트리 수 (rounds 없으면 2018 경기 20%로 조기 종료)
    if rounds is None:
        t_, v_ = itr[~es], itr[es]
        dtr = lgb.Dataset(XLOC[cols].iloc[t_], y_zone[t_], init_score=INIT_L[t_], free_raw_data=False)
        dva = lgb.Dataset(XLOC[cols].iloc[v_], y_zone[v_], init_score=INIT_L[v_], reference=dtr)
        bst = lgb.train({**BASE_L, **LOC_LGB18}, dtr, 3000, valid_sets=[dva], callbacks=[lgb.early_stopping(100, verbose=False)])
        n = bst.best_iteration
    else:
        bst = lgb.train({**BASE_L, **LOC_LGB18}, lgb.Dataset(XLOC[cols].iloc[itr], y_zone[itr], init_score=INIT_L[itr]), rounds)
        n = rounds
    raw = bst.predict(XLOC[cols].iloc[i19], raw_score=True, num_iteration=n) + INIT_L[i19]
    return np.exp(log_softmax(raw, axis=1)), n

y19 = y_zone[i19]
oc19 = (fc.ZONE_GROUP[y19] == "C").astype(float)
nll19 = lambda P: -np.log(np.clip(P[np.arange(len(i19)), y19], 1e-12, None))

def game_ci(d, B=1000):  # 경기 단위 부트스트랩 95% 구간 (2019)
    s = pd.DataFrame({"g": gid_all[i19], "d": d}).groupby("g")["d"].agg(["sum", "count"])
    i = np.random.default_rng(0).integers(0, len(s), (B, len(s)))
    return np.percentile(s["sum"].to_numpy()[i].sum(1) / s["count"].to_numpy()[i].sum(1), [2.5, 97.5])

def auc(p, o):  # Mann-Whitney AUC
    r = pd.Series(p).rank().to_numpy()
    n1, n0 = o.sum(), len(o) - o.sum()
    return (r[o == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

P18, _ = fit_pred(LOC_FEAT18, LOC_ROUNDS18)
assert np.allclose(nll19(P18), fc.align(pp18, fp, fc.VAL_YEAR)["ll_zone_true"].to_numpy(), atol=1e-6), "model_14 재현 실패"
print(f"model_14 2019 24칸 log loss (실제 구종) {nll19(P18).mean():.5f} (재현 확인)")
rows = []
for mode, rounds in [("트리 168 고정", LOC_ROUNDS18), ("조기 종료", None)]:
    Pb, nb_ = (P18, LOC_ROUNDS18) if rounds else fit_pred(LOC_FEAT18)
    llb = nll19(Pb)
    rows.append({"방식": mode, "더한 변수": "(model_14 구조)", "트리 수": nb_, "2019 24칸 ll": llb.mean(), "이득 ×1000": 0.0,
                 "존 밖 AUC": auc(Pb[:, fc.ZONE_GROUP == "C"].sum(1), oc19)})
    for nm, cols in GROUPS.items():
        P, n_ = fit_pred(LOC_FEAT18 + cols, rounds)
        d = llb - nll19(P)
        lo, hi = game_ci(d)
        rows.append({"방식": mode, "더한 변수": nm, "트리 수": n_, "2019 24칸 ll": nll19(P).mean(), "이득 ×1000": d.mean() * 1000,
                     "95% 하한": lo * 1000, "95% 상한": hi * 1000, "존 밖 AUC": auc(P[:, fc.ZONE_GROUP == "C"].sum(1), oc19)})
display(pd.DataFrame(rows).set_index(["방식", "더한 변수"]).round(4))

model_14 2019 24칸 log loss (실제 구종) 2.88354 (재현 확인)


트리 수  2019 24칸 ll  이득 ×1000  존 밖 AUC  95% 하한  95% 상한
방식        더한 변수                                                              
트리 168 고정 (model_14 구조)   168       2.8835    0.0000   0.5546     NaN     NaN
          포수·구장·월 전부      168       2.8844   -0.8272   0.5538 -2.1947  0.5344
          포수              168       2.8845   -0.9625   0.5535 -2.0742  0.1564
          구장              168       2.8844   -0.9028   0.5533 -1.9150  0.1686
          월               168       2.8834    0.1602   0.5549 -0.3109  0.5747
조기 종료     (model_14 구조)   201       2.8832    0.0000   0.5557     NaN     NaN
          포수·구장·월 전부      148       2.8841   -0.9029   0.5524 -2.2703  0.4012
          포수              194       2.8840   -0.7991   0.5543 -1.9138  0.2619
          구장              107       2.8844   -1.1694   0.5515 -2.3737 -0.0907
          월               190       2.8833   -0.1357   0.5555 -0.6348  0.3262

#### 🔍 0-1 결과: model_14 위 포수·구장·월은 남는 몫 없음, 포수·구장 잔차는 오히려 나빠지는 쪽
| 더한 변수 (이득 ×1000, 2019) | 트리 168 고정 (95%) | 조기 종료 (95%) |
| --- | --- | --- |
| 포수·구장·월 전부 | −0.83 (−2.19 ~ +0.53) | −0.90 (−2.27 ~ +0.40) |
| 포수 잔차 | −0.96 (−2.07 ~ +0.16) | −0.80 (−1.91 ~ +0.26) |
| 구장 잔차 | −0.90 (−1.92 ~ +0.17) | **−1.17** (−2.37 ~ −0.09) |
| 월 | +0.16 (−0.31 ~ +0.57) | −0.14 (−0.63 ~ +0.33) |

- model_14 재현 확인 (2019 2.88354)
- 4-28 사전 확인의 +0.60(상황 없이)은 model_14 위에서 사라짐, 포수·구장 잔차는 두 방식 모두 − 쪽 (구장은 조기 종료에서 95% 구간 0 아래)
- 구종 4-16과 같은 모습: 포수·구장 잔차는 그 포수와 짝인 투수 / 그 구장 홈 팀 투수의 쏠림이라 다음 해에 반복 안 됨
- 월은 0 근처 (시즌 초 효과는 model_11 시즌 초 배수가 가져감)


### 0-2. model_14 위 직전 타자 결과

- 0-1과 같은 방식 (model_14 구조, 학습 2018 -> 2019, 트리 168 고정 / 조기 종료)
- 변수: `fc.game_flow_features` (경기 기록, 그 타석 앞 정보, 미래 정보 검사 통과 4-13)
  - `prev_first_*` 바로 앞 타석(다른 타자) 초구 결과 원-핫 (없음 = 그 경기 첫 타자 / 볼 / 루킹 / 헛스윙 / 파울 / 인플레이)
  - `prev_how_*` 바로 앞 타석 결과 원-핫 (없음 / 안타 / 홈런 / 볼넷·사구 / 삼진 / 아웃·기타)
- 4-28 사전 확인: 하나만 더함 −0.25 (0 포함), 전부에서 빼면 +0.71 (0 위) / 구종 model_07에서는 앞 타석 초구 결과(인플레이 -> 다음 타자 직구↑)를 채택


In [3]:
FLOW = fc.game_flow_features(m, fp)
PF = ["prev_first_" + c for c in fc.FIRST_CATS]
PH = ["prev_how_" + c for c in fc.HOW_CATS]
XLOC[PF + PH] = FLOW[PF + PH]
rows = []
for mode, rounds in [("트리 168 고정", LOC_ROUNDS18), ("조기 종료", None)]:
    Pb, nb_ = (P18, LOC_ROUNDS18) if rounds else fit_pred(LOC_FEAT18)
    llb = nll19(Pb)
    rows.append({"방식": mode, "더한 변수": "(model_14 구조)", "트리 수": nb_, "2019 24칸 ll": llb.mean(), "이득 ×1000": 0.0,
                 "존 밖 AUC": auc(Pb[:, fc.ZONE_GROUP == "C"].sum(1), oc19)})
    for nm, cols in {"직전 타자 결과 전부": PF + PH, "앞 타석 초구 결과": PF, "앞 타석 결과": PH}.items():
        P, n_ = fit_pred(LOC_FEAT18 + cols, rounds)
        d = llb - nll19(P)
        lo, hi = game_ci(d)
        rows.append({"방식": mode, "더한 변수": nm, "트리 수": n_, "2019 24칸 ll": nll19(P).mean(), "이득 ×1000": d.mean() * 1000,
                     "95% 하한": lo * 1000, "95% 상한": hi * 1000, "존 밖 AUC": auc(P[:, fc.ZONE_GROUP == "C"].sum(1), oc19)})
display(pd.DataFrame(rows).set_index(["방식", "더한 변수"]).round(4))


트리 수  2019 24칸 ll  이득 ×1000  존 밖 AUC  95% 하한  95% 상한
방식        더한 변수                                                              
트리 168 고정 (model_14 구조)   168       2.8835    0.0000   0.5546     NaN     NaN
          직전 타자 결과 전부     168       2.8835    0.0107   0.5546 -0.5778  0.6039
          앞 타석 초구 결과      168       2.8838   -0.3034   0.5542 -0.7870  0.1530
          앞 타석 결과         168       2.8835    0.0742   0.5550 -0.4283  0.5447
조기 종료     (model_14 구조)   201       2.8832    0.0000   0.5557     NaN     NaN
          직전 타자 결과 전부     214       2.8834   -0.1687   0.5564 -0.8381  0.5104
          앞 타석 초구 결과      189       2.8836   -0.4219   0.5551 -0.9621  0.0666
          앞 타석 결과         214       2.8831    0.0979   0.5567 -0.4393  0.6192

#### 🔍 0-2 결과: model_14 위 직전 타자 결과는 남는 몫 없음
| 더한 변수 (이득 ×1000, 2019) | 트리 168 고정 (95%) | 조기 종료 (95%) |
| --- | --- | --- |
| 직전 타자 결과 전부 | +0.01 (−0.58 ~ +0.60) | −0.17 (−0.84 ~ +0.51) |
| 앞 타석 초구 결과 | −0.30 (−0.79 ~ +0.15) | −0.42 (−0.96 ~ +0.07) |
| 앞 타석 결과 | +0.07 (−0.43 ~ +0.54) | +0.10 (−0.44 ~ +0.62) |

- 모두 0 근처, 95% 구간이 0을 포함
- 4-28의 "전부에서 빼면 +0.71"은 다른 묶음과 함께일 때만 보였던 몫 -> 상황이 들어간 model_14 위에서는 없음
- 구종에서는 앞 타석 초구 인플레이 -> 다음 타자 직구가 남았지만 (4-13), 위치에는 이어지지 않음


### 0-3. model_14 위 투수×상황 위치 반응

- 0-1과 같은 방식 (model_14 구조, 학습 2018 -> 2019, 트리 168 고정 / 조기 종료)
- 변수: 투수×상황 칸별 H/S/C 잔차 (`fc.group_residual_loc`, 전날까지 (실제 − model_13 확률) 합 / (공 수 + k))
  - = 그 투수가 그 상황에서 지금 위치 확률보다 존 안/밖으로 얼마나 치우쳤나 (예: 투수별 득점권 피해 가기)
  - 잔차 기준은 model_13 (상황 보정 전, 4-28와 같음) -> 트리가 상황 변수와 함께 써서 "리그 평균 상황 효과와 다른 몫"을 배울 수 있음
- 칸 표본이 적은 문제 (구종 4-14) 때문에 여러 가지로
  - 4칸 (첫 타자 / 득점권 / 재대결 / 그 밖, 4-28와 같음) · k 50 / k 200 (더 세게 0 쪽으로)
  - 2칸 (득점권 / 아님) · k 50 -> 칸당 표본이 큼


In [4]:
P3 = PZ_TRUE @ fc.GROUP2  # model_13 H/S/C 확률
risp = (SIT_L["base2"] + SIT_L["base3"]) > 0
fpx = fp.assign(sit4=np.select([SIT_L["pc"].eq(0), risp, SIT_L["tto"].ge(2)], ["첫타자", "득점권", "재대결"], "그밖"), sit2=np.where(risp, "득점권", "아님"))
VARS = {}
for nm, cell, k in [("4칸 · k 50", "sit4", 50.0), ("4칸 · k 200", "sit4", 200.0), ("득점권 2칸 · k 50", "sit2", 50.0)]:
    R = fc.group_residual_loc(fpx, P3, ["pitcher_code", cell], fpx, k=k, prefix=f"ps_{cell}_{int(k)}")
    XLOC[R.columns] = R
    VARS[nm] = list(R.columns)
rows = []
for mode, rounds in [("트리 168 고정", LOC_ROUNDS18), ("조기 종료", None)]:
    Pb, nb_ = (P18, LOC_ROUNDS18) if rounds else fit_pred(LOC_FEAT18)
    llb = nll19(Pb)
    rows.append({"방식": mode, "더한 변수": "(model_14 구조)", "트리 수": nb_, "2019 24칸 ll": llb.mean(), "이득 ×1000": 0.0,
                 "존 밖 AUC": auc(Pb[:, fc.ZONE_GROUP == "C"].sum(1), oc19)})
    for nm, cols in VARS.items():
        P, n_ = fit_pred(LOC_FEAT18 + cols, rounds)
        d = llb - nll19(P)
        lo, hi = game_ci(d)
        rows.append({"방식": mode, "더한 변수": nm, "트리 수": n_, "2019 24칸 ll": nll19(P).mean(), "이득 ×1000": d.mean() * 1000,
                     "95% 하한": lo * 1000, "95% 상한": hi * 1000, "존 밖 AUC": auc(P[:, fc.ZONE_GROUP == "C"].sum(1), oc19)})
display(pd.DataFrame(rows).set_index(["방식", "더한 변수"]).round(4))
# 칸 표본: 2019 공마다 그 투수×칸의 전날까지 초구 수
for cell in ["sit4", "sit2"]:
    _, N = fc.lookup(fc.prior_table(fpx.assign(_one=1.0), ["pitcher_code", cell], ["_one"], cache=False), fpx[val], ["pitcher_code", cell], ["_one"])
    print(f"{cell}: 2019 공마다 투수×칸 전날까지 초구 수 중앙값 {np.median(N[:, 0]):.0f}, 50개 미만 비율 {(N[:, 0] < 50).mean():.2f}")


트리 수  2019 24칸 ll  이득 ×1000  존 밖 AUC  95% 하한  95% 상한
방식        더한 변수                                                              
트리 168 고정 (model_14 구조)   168       2.8835    0.0000   0.5546     NaN     NaN
          4칸 · k 50       168       2.8847   -1.1823   0.5535 -2.1676 -0.2134
          4칸 · k 200      168       2.8850   -1.5052   0.5540 -2.6215 -0.3929
          득점권 2칸 · k 50   168       2.8839   -0.3593   0.5537 -1.4680  0.7631
조기 종료     (model_14 구조)   201       2.8832    0.0000   0.5557     NaN     NaN
          4칸 · k 50       117       2.8845   -1.3101   0.5526 -2.4885 -0.1852
          4칸 · k 200      141       2.8845   -1.3226   0.5539 -2.5150 -0.1677
          득점권 2칸 · k 50   137       2.8833   -0.1568   0.5528 -1.3082  0.9816

sit4: 2019 공마다 투수×칸 전날까지 초구 수 중앙값 252, 50개 미만 비율 0.07
sit2: 2019 공마다 투수×칸 전날까지 초구 수 중앙값 408, 50개 미만 비율 0.03


#### 🔍 0-3 결과: model_14 위 투수×상황 반응은 오히려 나빠짐
| 더한 변수 (이득 ×1000, 2019) | 트리 168 고정 (95%) | 조기 종료 (95%) |
| --- | --- | --- |
| 4칸 · k 50 | **−1.18** (−2.17 ~ −0.21) | **−1.31** (−2.49 ~ −0.19) |
| 4칸 · k 200 | **−1.51** (−2.62 ~ −0.39) | **−1.32** (−2.52 ~ −0.17) |
| 득점권 2칸 · k 50 | −0.36 (−1.47 ~ +0.76) | −0.16 (−1.31 ~ +0.98) |

- 4칸은 두 방식·두 k 모두 95% 구간이 0 아래 (나빠짐), 칸을 2개로 줄여도 0 근처
- 칸 표본은 아주 적지 않음 (2019 공마다 투수×칸 전날까지 초구 중앙값 4칸 252 · 2칸 408) -> 표본 부족만의 문제는 아님
- 4-23: 투수별 위치 치우침은 해마다 안 이어짐 (상관 −0.28 ~ 0.18) -> 투수×상황 치우침도 학습 해 패턴을 배우는 것으로 보임


### 0-4. model_14 위 리그 최근 위치 잔차

- 0-1과 같은 방식 (model_14 구조, 학습 2018 -> 2019, 트리 168 고정 / 조기 종료)
- 변수: 분석 대상 투수 전체 초구의 최근 H/S/C 잔차 (`fc.group_residual_loc` 그룹 = 전체 하나, 전날까지 반감기 가중 (실제 − model_13 확률) 합 / (공 수 + 50))
  - = 요즘 리그 전체가 지금 위치 확률보다 존 안/밖으로 얼마나 치우쳐 있나 (4-23: 2019 높이·C 추세, 구종 4-20 방식)
  - 반감기 세 가지 (공 수): 500 (약 1주) / 1500 (약 3주) / 5000 (약 두 달), 4-28는 500
- 구종 4-20는 2019에 신호 없음 (2020에만 약하게)


In [5]:
LG = {}
for hl in [500.0, 1500.0, 5000.0]:
    R = fc.group_residual_loc(fpx.assign(_all=1), P3, ["_all"], fpx.assign(_all=1), half_life=hl, prefix=f"lg_{int(hl)}")
    XLOC[R.columns] = R
    LG[f"반감 {int(hl)}구"] = list(R.columns)
print("2019 리그 최근 C 잔차 (반감 1500) 범위:", XLOC.loc[val, "lg_1500_C"].describe()[["mean", "min", "max"]].round(4).to_dict())
rows = []
for mode, rounds in [("트리 168 고정", LOC_ROUNDS18), ("조기 종료", None)]:
    Pb, nb_ = (P18, LOC_ROUNDS18) if rounds else fit_pred(LOC_FEAT18)
    llb = nll19(Pb)
    rows.append({"방식": mode, "더한 변수": "(model_14 구조)", "트리 수": nb_, "2019 24칸 ll": llb.mean(), "이득 ×1000": 0.0,
                 "존 밖 AUC": auc(Pb[:, fc.ZONE_GROUP == "C"].sum(1), oc19)})
    for nm, cols in LG.items():
        P, n_ = fit_pred(LOC_FEAT18 + cols, rounds)
        d = llb - nll19(P)
        lo, hi = game_ci(d)
        rows.append({"방식": mode, "더한 변수": nm, "트리 수": n_, "2019 24칸 ll": nll19(P).mean(), "이득 ×1000": d.mean() * 1000,
                     "95% 하한": lo * 1000, "95% 상한": hi * 1000, "존 밖 AUC": auc(P[:, fc.ZONE_GROUP == "C"].sum(1), oc19)})
display(pd.DataFrame(rows).set_index(["방식", "더한 변수"]).round(4))


2019 리그 최근 C 잔차 (반감 1500) 범위: {'mean': -0.0143, 'min': -0.0255, 'max': -0.0011}


트리 수  2019 24칸 ll  이득 ×1000  존 밖 AUC  95% 하한  95% 상한
방식        더한 변수                                                              
트리 168 고정 (model_14 구조)   168       2.8835    0.0000   0.5546     NaN     NaN
          반감 500구         168       2.8848   -1.2266   0.5552 -2.2607 -0.1870
          반감 1500구        168       2.8847   -1.1516   0.5546 -2.1606 -0.1273
          반감 5000구        168       2.8858   -2.2614   0.5540 -3.2949 -1.2348
조기 종료     (model_14 구조)   201       2.8832    0.0000   0.5557     NaN     NaN
          반감 500구         148       2.8844   -1.2596   0.5542 -2.3283 -0.2354
          반감 1500구        137       2.8844   -1.2582   0.5539 -2.4388 -0.1930
          반감 5000구        118       2.8846   -1.4036   0.5532 -2.5277 -0.3478

#### 🔍 0-4 결과: model_14 위 리그 최근 위치 잔차는 나빠짐
| 더한 변수 (이득 ×1000, 2019) | 트리 168 고정 (95%) | 조기 종료 (95%) |
| --- | --- | --- |
| 반감 500구 | **−1.23** (−2.26 ~ −0.19) | **−1.26** (−2.33 ~ −0.24) |
| 반감 1500구 | **−1.15** (−2.16 ~ −0.13) | **−1.26** (−2.44 ~ −0.19) |
| 반감 5000구 | **−2.26** (−3.29 ~ −1.23) | **−1.40** (−2.53 ~ −0.35) |

- 반감기와 상관없이 모두 95% 구간이 0 아래
- 2019 리그 최근 C 잔차 (반감 1500)는 시즌 내내 음수 (평균 −0.014, −0.026 ~ −0.001) -> 학습 해(2018) 값과 범위가 달라 학습 해에서 배운 규칙이 안 맞음 (해마다 범위가 바뀌는 변수, 2단계 bat_n과 같은 문제)
- 구종 4-20와 같은 결론: 리그 최근 어긋남으로 다음 어긋남을 미리 알 수 없음


### 0-5. model_14 위 시즌 등판 순번

- 0-1과 같은 방식 (model_14 구조, 학습 2018 -> 2019, 트리 168 고정 / 조기 종료)
- 근거: 4-23 시즌 첫 1–3등판에서 존 밖(C) 과다 예측 (rC 2019 −0.071 / 2020 −0.036), 시즌 초 배수(model_11)로 안 고쳐짐 (4-24)
- 변수 (`fc.appearance_features`, 경기 기록, 미래 정보 검사 통과 4-28) 세 가지 형태
  - 등판 순번 그대로 (`app_no`, 1–71) / 10에서 자름 (해마다 범위가 비슷하게) / 첫 등판·2–3번째 표시 (원-핫 2개)
- 진단: 시즌 첫 등판·2–3번째 등판의 존 밖 실제 − 예측 (model_14 vs 더한 모델)


In [6]:
APP = fc.appearance_features(m, fp)["app_no"]
XLOC["app_no"] = APP
XLOC["app_no10"] = APP.clip(upper=10)
XLOC["app_first"], XLOC["app_2_3"] = APP.eq(1).astype(int), APP.between(2, 3).astype(int)
APPV = {"등판 순번 그대로": ["app_no"], "10에서 자름": ["app_no10"], "첫 등판·2–3번째 표시": ["app_first", "app_2_3"]}
rows, Pkeep = [], {}
for mode, rounds in [("트리 168 고정", LOC_ROUNDS18), ("조기 종료", None)]:
    Pb, nb_ = (P18, LOC_ROUNDS18) if rounds else fit_pred(LOC_FEAT18)
    llb = nll19(Pb)
    rows.append({"방식": mode, "더한 변수": "(model_14 구조)", "트리 수": nb_, "2019 24칸 ll": llb.mean(), "이득 ×1000": 0.0,
                 "존 밖 AUC": auc(Pb[:, fc.ZONE_GROUP == "C"].sum(1), oc19)})
    for nm, cols in APPV.items():
        P, n_ = fit_pred(LOC_FEAT18 + cols, rounds)
        Pkeep[(mode, nm)] = P
        d = llb - nll19(P)
        lo, hi = game_ci(d)
        rows.append({"방식": mode, "더한 변수": nm, "트리 수": n_, "2019 24칸 ll": nll19(P).mean(), "이득 ×1000": d.mean() * 1000,
                     "95% 하한": lo * 1000, "95% 상한": hi * 1000, "존 밖 AUC": auc(P[:, fc.ZONE_GROUP == "C"].sum(1), oc19)})
display(pd.DataFrame(rows).set_index(["방식", "더한 변수"]).round(4))
grp = pd.cut(APP[val].to_numpy(), [1, 2, 4, 11, 999], right=False, labels=["1", "2–3", "4–10", "11+"])
cmp = {"model_14": P18, **{nm: Pkeep[("트리 168 고정", nm)] for nm in APPV}}
tab = pd.DataFrame({nm: pd.Series(oc19 - P[:, fc.ZONE_GROUP == "C"].sum(1)).groupby(np.asarray(grp)).mean() for nm, P in cmp.items()})
tab.insert(0, "공", pd.Series(np.asarray(grp)).value_counts())
print("2019 시즌 몇 번째 등판별 존 밖(C) 실제 − 예측 (트리 168 고정)")
display(tab.round(4))


트리 수  2019 24칸 ll  이득 ×1000  존 밖 AUC  95% 하한  95% 상한
방식        더한 변수                                                              
트리 168 고정 (model_14 구조)   168       2.8835    0.0000   0.5546     NaN     NaN
          등판 순번 그대로       168       2.8838   -0.2606   0.5543 -1.0372  0.4145
          10에서 자름         168       2.8840   -0.4132   0.5544 -0.9727  0.0871
          첫 등판·2–3번째 표시   168       2.8836   -0.0833   0.5546 -0.2749  0.1346
조기 종료     (model_14 구조)   201       2.8832    0.0000   0.5557     NaN     NaN
          등판 순번 그대로       185       2.8836   -0.4391   0.5553 -1.1357  0.2113
          10에서 자름         214       2.8837   -0.4857   0.5554 -1.0317  0.0138
          첫 등판·2–3번째 표시   205       2.8831    0.0815   0.5557 -0.1525  0.3147

2019 시즌 몇 번째 등판별 존 밖(C) 실제 − 예측 (트리 168 고정)


,공,model_14,등판 순번 그대로,10에서 자름,첫 등판·2–3번째 표시
1,419,-0.0578,-0.0664,-0.0659,-0.0626
11+,8741,-0.0105,-0.0072,-0.0077,-0.0099
2–3,925,0.0003,-0.0074,-0.0077,-0.0032
4–10,3373,-0.0118,-0.0142,-0.0137,-0.0111


#### 🔍 0-5 결과: model_14 위 시즌 등판 순번은 남는 몫 없음, 첫 등판 C 과다 예측도 안 고쳐짐
| 더한 변수 (이득 ×1000, 2019) | 트리 168 고정 (95%) | 조기 종료 (95%) |
| --- | --- | --- |
| 등판 순번 그대로 | −0.26 (−1.04 ~ +0.41) | −0.44 (−1.14 ~ +0.21) |
| 10에서 자름 | −0.41 (−0.97 ~ +0.09) | −0.49 (−1.03 ~ +0.01) |
| 첫 등판·2–3번째 표시 | −0.08 (−0.27 ~ +0.13) | +0.08 (−0.15 ~ +0.31) |

- 모두 0 근처 또는 − 쪽, 95% 구간이 0을 포함
- 2019 시즌 첫 등판 존 밖 실제 − 예측: model_14 −0.058 -> 더해도 −0.063 ~ −0.066 (오히려 조금 커짐) -> 학습 해(2018)에서 배운 첫 등판 효과가 2019에 반복되지 않음
- 시즌 첫 등판 C 과다 예측(4-23)은 시즌 초 배수(4-24)로도, 등판 순번 변수로도 안 잡힘 -> 해마다 다른 일시적인 모습으로 보임


### 0-6. model_14 위 이 타자와 앞 대결

- 0-1과 같은 방식 (model_14 구조, 학습 2018 -> 2019, 트리 168 고정 / 조기 종료)
- 변수 (경기 기록, 그 타석 앞 정보)
  - `tto` 같은 타자 몇 번째 대결 (`fc.situation_features`, model_14에는 안 들어 있음)
  - `sb_prev_*` 그 경기 이 타자와 바로 앞 대결 결과 원-핫 (없음 / 안타 / 장타 / 볼넷·사구 / 삼진 / 아웃·기타, `fc.same_batter_features`, 미래 정보 검사 통과 4-28)
- 생각한 효과: 앞 타석에 장타를 맞았으면 피해 가는 승부 (존 밖), 삼진을 잡았으면 같은 공략
- 4-28 사전 확인: 하나만 더함 −0.05, 전부에서 빼면 +0.02 (둘 다 0 근처)


In [7]:
SBF = fc.same_batter_features(m, fp)
SBC = ["sb_prev_" + c for c in fc.SB_CATS]
XLOC[SBC] = SBF[SBC]
XLOC["tto"] = SIT_L["tto"]
SBV = {"앞 대결 전부 (몇 번째 + 결과)": ["tto"] + SBC, "같은 타자 몇 번째": ["tto"], "앞 대결 결과": SBC}
rows = []
for mode, rounds in [("트리 168 고정", LOC_ROUNDS18), ("조기 종료", None)]:
    Pb, nb_ = (P18, LOC_ROUNDS18) if rounds else fit_pred(LOC_FEAT18)
    llb = nll19(Pb)
    rows.append({"방식": mode, "더한 변수": "(model_14 구조)", "트리 수": nb_, "2019 24칸 ll": llb.mean(), "이득 ×1000": 0.0,
                 "존 밖 AUC": auc(Pb[:, fc.ZONE_GROUP == "C"].sum(1), oc19)})
    for nm, cols in SBV.items():
        P, n_ = fit_pred(LOC_FEAT18 + cols, rounds)
        d = llb - nll19(P)
        lo, hi = game_ci(d)
        rows.append({"방식": mode, "더한 변수": nm, "트리 수": n_, "2019 24칸 ll": nll19(P).mean(), "이득 ×1000": d.mean() * 1000,
                     "95% 하한": lo * 1000, "95% 상한": hi * 1000, "존 밖 AUC": auc(P[:, fc.ZONE_GROUP == "C"].sum(1), oc19)})
display(pd.DataFrame(rows).set_index(["방식", "더한 변수"]).round(4))
# 진단: 앞 대결 결과별 존 밖 실제 − model_14 예측 (2019) -> 신호가 있는지
prev = SBF.loc[val, SBC].to_numpy().argmax(1)
d = pd.DataFrame({"앞 대결": np.array(fc.SB_CATS)[prev], "r": oc19 - P18[:, fc.ZONE_GROUP == "C"].sum(1)})
t = d.groupby("앞 대결")["r"].agg(["size", "mean", "std"])
t["se"] = t["std"] / np.sqrt(t["size"])
print("2019 이 타자와 앞 대결 결과별 존 밖(C) 실제 − model_14 예측")
display(t[["size", "mean", "se"]].rename(columns={"size": "공", "mean": "rC"}).round(4))


트리 수  2019 24칸 ll  이득 ×1000  존 밖 AUC  95% 하한  95% 상한
방식        더한 변수                                                                    
트리 168 고정 (model_14 구조)         168       2.8835    0.0000   0.5546     NaN     NaN
          앞 대결 전부 (몇 번째 + 결과)   168       2.8837   -0.1613   0.5546 -0.5796  0.2714
          같은 타자 몇 번째            168       2.8837   -0.1833   0.5546 -0.5037  0.1397
          앞 대결 결과               168       2.8837   -0.2069   0.5546 -0.6007  0.1469
조기 종료     (model_14 구조)         201       2.8832    0.0000   0.5557     NaN     NaN
          앞 대결 전부 (몇 번째 + 결과)   210       2.8837   -0.4812   0.5553 -1.0208  0.0026
          같은 타자 몇 번째            190       2.8836   -0.3896   0.5551 -0.7701 -0.0313
          앞 대결 결과               210       2.8837   -0.4964   0.5550 -0.9392 -0.0768

2019 이 타자와 앞 대결 결과별 존 밖(C) 실제 − model_14 예측


,공,rC,se
앞 대결,,,
볼넷·사구,480,-0.0514,0.0219
삼진,930,0.0214,0.0162
아웃·기타,2973,-0.0155,0.0089
안타,1033,0.0098,0.0152
없음,7645,-0.0158,0.0055
장타,397,0.0135,0.0247


#### 🔍 0-6 결과: model_14 위 이 타자와 앞 대결은 남는 몫 없음
| 더한 변수 (이득 ×1000, 2019) | 트리 168 고정 (95%) | 조기 종료 (95%) |
| --- | --- | --- |
| 앞 대결 전부 (몇 번째 + 결과) | −0.16 (−0.58 ~ +0.27) | −0.48 (−1.02 ~ +0.00) |
| 같은 타자 몇 번째 | −0.18 (−0.50 ~ +0.14) | −0.39 (−0.77 ~ −0.03) |
| 앞 대결 결과 | −0.21 (−0.60 ~ +0.15) | −0.50 (−0.94 ~ −0.08) |

- 0 근처 또는 − 쪽 (조기 종료는 몇 번째·결과 하나씩이 95% 구간 0 아래)
- 앞 대결 결과별 존 밖 실제 − model_14 예측 (2019): 장타 뒤 +0.014 (표준오차 0.025) -> "장타 맞으면 피해 감" 신호 없음
  - 볼넷·사구 뒤 −0.051 (표준오차 0.022, 480개): 볼넷 뒤에는 존 안으로 더 던지는 쪽이지만 공이 적고 6칸 중 하나라 우연일 수 있음


### 0-7. model_14 위 투수 상태

- 0-1과 같은 방식 (model_14 구조, 학습 2018 -> 2019, 트리 168 고정 / 조기 종료)
- 변수 (경기 기록, 그 공 앞 정보)
  - `pc` 그 경기 앞선 투구 수 (피로 -> 제구), `starter` 선발/불펜 (`fc.situation_features`)
  - `rest` 등판 간격(일), `role_change` 직전 등판과 선발/불펜이 다름 (`fc.pitcher_history_features`, 미래 정보 검사 통과 4-11)
- 4-28 사전 확인: 하나만 더함 −0.09, 전부에서 빼면 −0.03 (둘 다 0 근처) / 4-23: 앞선 투구 수 0(첫 타자) rC −0.033 (2020 −0.005)
- 진단: 앞선 투구 수 구간별 존 밖 실제 − model_14 예측


In [8]:
HISTF = fc.pitcher_history_features(m, fp)[["rest", "role_change"]]
XLOC[["rest", "role_change"]] = HISTF
XLOC["pc"], XLOC["starter"] = SIT_L["pc"], SIT_L["starter"]
PSV = {"투수 상태 전부": ["pc", "starter", "rest", "role_change"], "앞선 투구 수": ["pc"], "선발/불펜": ["starter"], "등판 간격·역할 전환": ["rest", "role_change"]}
rows = []
for mode, rounds in [("트리 168 고정", LOC_ROUNDS18), ("조기 종료", None)]:
    Pb, nb_ = (P18, LOC_ROUNDS18) if rounds else fit_pred(LOC_FEAT18)
    llb = nll19(Pb)
    rows.append({"방식": mode, "더한 변수": "(model_14 구조)", "트리 수": nb_, "2019 24칸 ll": llb.mean(), "이득 ×1000": 0.0,
                 "존 밖 AUC": auc(Pb[:, fc.ZONE_GROUP == "C"].sum(1), oc19)})
    for nm, cols in PSV.items():
        P, n_ = fit_pred(LOC_FEAT18 + cols, rounds)
        d = llb - nll19(P)
        lo, hi = game_ci(d)
        rows.append({"방식": mode, "더한 변수": nm, "트리 수": n_, "2019 24칸 ll": nll19(P).mean(), "이득 ×1000": d.mean() * 1000,
                     "95% 하한": lo * 1000, "95% 상한": hi * 1000, "존 밖 AUC": auc(P[:, fc.ZONE_GROUP == "C"].sum(1), oc19)})
display(pd.DataFrame(rows).set_index(["방식", "더한 변수"]).round(4))
pcg = pd.cut(SIT_L.loc[val, "pc"].to_numpy(), [0, 1, 16, 31, 61, 91, 999], right=False, labels=["0 (첫 타자)", "1–15", "16–30", "31–60", "61–90", "91+"])
d = pd.DataFrame({"앞선 투구 수": np.asarray(pcg), "선발": SIT_L.loc[val, "starter"].map({1: "선발", 0: "불펜"}).to_numpy(),
                  "r": oc19 - P18[:, fc.ZONE_GROUP == "C"].sum(1)})
for var in ["앞선 투구 수", "선발"]:
    t = d.groupby(var)["r"].agg(["size", "mean", "std"])
    t["se"] = t["std"] / np.sqrt(t["size"])
    print(f"2019 {var}별 존 밖(C) 실제 − model_14 예측")
    display(t[["size", "mean", "se"]].rename(columns={"size": "공", "mean": "rC"}).round(4))


트리 수  2019 24칸 ll  이득 ×1000  존 밖 AUC  95% 하한  95% 상한
방식        더한 변수                                                              
트리 168 고정 (model_14 구조)   168       2.8835    0.0000   0.5546     NaN     NaN
          투수 상태 전부        168       2.8843   -0.7807   0.5542 -1.4853 -0.0063
          앞선 투구 수         168       2.8839   -0.3805   0.5545 -0.9857  0.2365
          선발/불펜           168       2.8837   -0.1401   0.5546 -0.3147  0.0540
          등판 간격·역할 전환     168       2.8841   -0.5519   0.5543 -0.9868 -0.0785
조기 종료     (model_14 구조)   201       2.8832    0.0000   0.5557     NaN     NaN
          투수 상태 전부        156       2.8842   -1.0367   0.5540 -1.9067 -0.1931
          앞선 투구 수         190       2.8840   -0.7802   0.5549 -1.4814 -0.1428
          선발/불펜           226       2.8836   -0.4021   0.5560 -0.7028 -0.0960
          등판 간격·역할 전환     152       2.8836   -0.3784   0.5546 -1.0036  0.2402

2019 앞선 투구 수별 존 밖(C) 실제 − model_14 예측


,공,rC,se
앞선 투구 수,,,
0 (첫 타자),1288,-0.0241,0.0134
16–30,2243,-0.0308,0.0102
1–15,3868,-0.0043,0.0078
31–60,3220,-0.0078,0.0086
61–90,2492,0.0012,0.0098
91+,347,-0.0482,0.0256


2019 선발별 존 밖(C) 실제 − model_14 예측


,공,rC,se
선발,,,
불펜,4242,-0.0122,0.0075
선발,9216,-0.0113,0.0051


#### 🔍 0-7 결과: model_14 위 투수 상태는 오히려 나빠지는 쪽
| 더한 변수 (이득 ×1000, 2019) | 트리 168 고정 (95%) | 조기 종료 (95%) |
| --- | --- | --- |
| 투수 상태 전부 | **−0.78** (−1.49 ~ −0.01) | **−1.04** (−1.91 ~ −0.19) |
| 앞선 투구 수 | −0.38 (−0.99 ~ +0.24) | **−0.78** (−1.48 ~ −0.14) |
| 선발/불펜 | −0.14 (−0.31 ~ +0.05) | **−0.40** (−0.70 ~ −0.10) |
| 등판 간격·역할 전환 | **−0.55** (−0.99 ~ −0.08) | −0.38 (−1.00 ~ +0.24) |

- 모두 − 쪽, 묶음 전체는 두 방식 모두 95% 구간이 0 아래
- 앞선 투구 수별 존 밖 실제 − model_14 예측 (2019): 구간마다 −0.048 ~ +0.001로 방향 없이 흩어짐 (91구 이상 −0.048은 347개) -> "피로하면 제구가 흔들려 존 밖" 같은 일관된 신호 없음, 선발/불펜도 같음 (−0.012 / −0.011)


### 0-8. model_14 위 오늘 제구

- 0-1과 같은 방식 (model_14 구조, 학습 2018 -> 2019, 트리 168 고정 / 조기 종료)
- 변수 (`fc.game_flow_features`, 경기 기록, 그 타석 앞 정보, 미래 정보 검사 통과 4-13)
  - `ball_rate` 그 경기 앞선 공 중 볼 비율 (앞 공 없으면 빈 값), `fs_rate` 그 경기 앞 타석들의 초구 스트라이크 비율
- 생각한 효과: 오늘 볼이 많으면 초구도 존 밖 가능성 큼
- 4-28 사전 확인: 하나만 더함 −0.58, 전부에서 빼면 −0.29 (둘 다 0 포함, − 쪽)
- 진단: 앞 공 볼 비율 구간별 존 밖 실제 − model_14 예측


In [9]:
FLOW8 = fc.game_flow_features(m, fp)
XLOC[["ball_rate", "fs_rate"]] = FLOW8[["ball_rate", "fs_rate"]]
TCV = {"오늘 제구 전부": ["ball_rate", "fs_rate"], "앞 공 볼 비율": ["ball_rate"], "앞 타석 초구 스트라이크 비율": ["fs_rate"]}
rows = []
for mode, rounds in [("트리 168 고정", LOC_ROUNDS18), ("조기 종료", None)]:
    Pb, nb_ = (P18, LOC_ROUNDS18) if rounds else fit_pred(LOC_FEAT18)
    llb = nll19(Pb)
    rows.append({"방식": mode, "더한 변수": "(model_14 구조)", "트리 수": nb_, "2019 24칸 ll": llb.mean(), "이득 ×1000": 0.0,
                 "존 밖 AUC": auc(Pb[:, fc.ZONE_GROUP == "C"].sum(1), oc19)})
    for nm, cols in TCV.items():
        P, n_ = fit_pred(LOC_FEAT18 + cols, rounds)
        d = llb - nll19(P)
        lo, hi = game_ci(d)
        rows.append({"방식": mode, "더한 변수": nm, "트리 수": n_, "2019 24칸 ll": nll19(P).mean(), "이득 ×1000": d.mean() * 1000,
                     "95% 하한": lo * 1000, "95% 상한": hi * 1000, "존 밖 AUC": auc(P[:, fc.ZONE_GROUP == "C"].sum(1), oc19)})
display(pd.DataFrame(rows).set_index(["방식", "더한 변수"]).round(4))
br = FLOW8.loc[val, "ball_rate"].to_numpy()
g = np.where(np.isnan(br), "앞 공 없음 (첫 타자)", pd.cut(br, [-0.01, 0.25, 0.35, 0.45, 1.01], labels=["~25%", "25–35%", "35–45%", "45%~"]).astype(str))
d = pd.DataFrame({"앞 공 볼 비율": g, "r": oc19 - P18[:, fc.ZONE_GROUP == "C"].sum(1)})
t = d.groupby("앞 공 볼 비율")["r"].agg(["size", "mean", "std"])
t["se"] = t["std"] / np.sqrt(t["size"])
print("2019 그 경기 앞 공 볼 비율별 존 밖(C) 실제 − model_14 예측")
display(t[["size", "mean", "se"]].rename(columns={"size": "공", "mean": "rC"}).round(4))


트리 수  2019 24칸 ll  이득 ×1000  존 밖 AUC  95% 하한  95% 상한
방식        더한 변수                                                                 
트리 168 고정 (model_14 구조)      168       2.8835    0.0000   0.5546     NaN     NaN
          오늘 제구 전부           168       2.8859   -2.3939   0.5528 -3.4106 -1.4521
          앞 공 볼 비율           168       2.8852   -1.6196   0.5532 -2.5247 -0.7848
          앞 타석 초구 스트라이크 비율   168       2.8849   -1.3126   0.5540 -2.1570 -0.5403
조기 종료     (model_14 구조)      201       2.8832    0.0000   0.5557     NaN     NaN
          오늘 제구 전부           227       2.8854   -2.2285   0.5552 -3.4559 -1.1582
          앞 공 볼 비율           210       2.8846   -1.4570   0.5554 -2.3941 -0.5601
          앞 타석 초구 스트라이크 비율   163       2.8845   -1.2849   0.5539 -2.0949 -0.4756

2019 그 경기 앞 공 볼 비율별 존 밖(C) 실제 − model_14 예측


,공,rC,se
앞 공 볼 비율,,,
25–35%,4070,-0.0088,0.0076
35–45%,4564,-0.0180,0.0072
45%~,1706,-0.0119,0.0118
~25%,1830,0.0074,0.0113
앞 공 없음 (첫 타자),1288,-0.0241,0.0134


#### 🔍 0-8 결과: model_14 위 오늘 제구는 뚜렷하게 나빠짐
| 더한 변수 (이득 ×1000, 2019) | 트리 168 고정 (95%) | 조기 종료 (95%) |
| --- | --- | --- |
| 오늘 제구 전부 | **−2.39** (−3.41 ~ −1.45) | **−2.23** (−3.46 ~ −1.16) |
| 앞 공 볼 비율 | **−1.62** (−2.52 ~ −0.78) | **−1.46** (−2.39 ~ −0.56) |
| 앞 타석 초구 스트라이크 비율 | **−1.31** (−2.16 ~ −0.54) | **−1.28** (−2.09 ~ −0.48) |

- 모두 두 방식에서 95% 구간이 0 아래
- 앞 공 볼 비율별 존 밖 실제 − model_14 예측 (2019): 25% 미만 +0.007, 25–35% −0.009, 35–45% −0.018, 45% 이상 −0.012 -> "오늘 볼이 많으면 초구도 존 밖" 신호 없음 (오히려 반대 쪽)
- 4-28 사전 확인과 같은 방향 (−0.58), 학습 해(2018)의 우연한 패턴을 배워 2019에 틀림

#### 🔍 0장 정리 (model_14 위에서 남은 묶음 다시 확인, 0-1 ~ 0-8)
- 포수·구장·월, 직전 타자 결과, 투수×상황, 리그 최근 잔차, 시즌 등판 순번, 이 타자와 앞 대결, 투수 상태, 오늘 제구: **모두 남는 몫 없음** (0 근처 또는 나빠짐)
- 4-28 사전 확인에서 + 였던 몫(포수·구장·월 +0.60, 직전 타자 결과 +0.71)은 상황과 겹친 것이었음
- 위치에서 쓸 수 있는 정보로 얻은 것은 상황(model_14)뿐 -> 0장 0-2 상한(실행 오차가 퍼짐 대부분)과 맞음


### 0-9. 상황 보정 재학습 주기: 월 단위

- 지금 (model_14): 상황 보정 LightGBM을 시즌에 한 번 학습 (2019 = 2018, 2020 = 2018–19)
- 구종 model_09처럼 **월 단위**: 달마다 그 달 첫 경기 전날까지 모든 초구(지난 시즌 + 그 시즌 지난달까지)로 다시 학습, 3월은 4월에·10월은 9월에 합침
  - 변수·파라미터·트리 수(168)는 model_14 그대로 (달마다 고르려면 미래 달이 필요)
  - 첫 달은 시즌 단위와 같은 모델 (재현 확인)
- 위치 보정은 학습이 2018 한 해뿐이라 그 시즌 데이터를 더 넣는 몫이 구종(4-19)보다 클 수 있음
- 이득 ×1000 = 시즌 단위 − 월 단위 24칸 log loss (실제 구종), 2019 판단 / 2020 확인, 경기 단위 95% 구간


In [10]:
for yr in (fc.TEST_YEAR,):  # 2020 model_13 위치 (실제 구종)
    k = fp["GYEAR"].eq(yr).to_numpy()
    PZ_TRUE[k] = loc_given_type(fp[k], yr, fp.loc[k, "pitch_type"].to_numpy(), **LOC17)[0]
INIT_L = np.log(np.clip(PZ_TRUE, 1e-6, None))
INIT_L -= INIT_L.mean(1, keepdims=True)
DATE_L = fp["game_date"].to_numpy()
MON_L = np.clip(fp["game_date"].dt.month.to_numpy(), 4, 9)

def fit_rows(tr):  # 학습 행 tr로 model_14 설정 (트리 168)
    return lgb.train({**BASE_L, **LOC_LGB18}, lgb.Dataset(XLOC[LOC_FEAT18].iloc[tr], y_zone[tr], init_score=INIT_L[tr]), LOC_ROUNDS18)

def pred_rows(bst, idx):
    return np.exp(log_softmax(bst.predict(XLOC[LOC_FEAT18].iloc[idx], raw_score=True) + INIT_L[idx], axis=1))

rows, plan = [], []
for yr in (fc.VAL_YEAR, fc.TEST_YEAR):
    iy = np.flatnonzero(yr_l == yr)
    yz = y_zone[iy]
    nll_ = lambda P: -np.log(np.clip(P[np.arange(len(iy)), yz], 1e-12, None))
    P_season = pred_rows(fit_rows(np.flatnonzero((yr_l >= 2018) & (yr_l < yr))), iy)  # 시즌 단위
    assert np.allclose(nll_(P_season), fc.align(pp18, fp, yr)["ll_zone_true"].to_numpy(), atol=1e-6), f"{yr} 시즌 단위 = model_14 재현 실패"
    P_month = np.full_like(P_season, np.nan)
    for mon in np.unique(MON_L[iy]):
        te = iy[MON_L[iy] == mon]
        start = DATE_L[te].min()
        tr = np.flatnonzero((yr_l >= 2018) & (DATE_L < start))
        assert DATE_L[tr].max() < start
        P_month[np.isin(iy, te)] = pred_rows(fit_rows(tr), te)
        plan.append({"연도": yr, "월": mon, "예측 초구": len(te), "학습 초구": len(tr), "그 시즌 학습 초구": int((yr_l[tr] == yr).sum())})
    d = nll_(P_season) - nll_(P_month)  # 양수 = 월 단위가 좋음
    lo, hi = np.percentile(pd.DataFrame({"g": gid_all[iy], "d": d}).groupby("g")["d"].agg(["sum", "count"]).pipe(
        lambda s: s["sum"].to_numpy()[np.random.default_rng(0).integers(0, len(s), (1000, len(s)))].sum(1) / s["count"].to_numpy()[np.random.default_rng(0).integers(0, len(s), (1000, len(s)))].sum(1)), [2.5, 97.5])
    oc = (fc.ZONE_GROUP[yz] == "C").astype(float)
    rows.append({"연도": yr, "시즌 단위 (model_14)": nll_(P_season).mean(), "월 단위": nll_(P_month).mean(), "이득 ×1000": d.mean() * 1000,
                 "95% 하한": lo * 1000, "95% 상한": hi * 1000,
                 "존 밖 AUC 시즌": auc(P_season[:, fc.ZONE_GROUP == "C"].sum(1), oc), "존 밖 AUC 월": auc(P_month[:, fc.ZONE_GROUP == "C"].sum(1), oc)})
    for mon in np.unique(MON_L[iy]):
        km = MON_L[iy] == mon
        plan[-len(np.unique(MON_L[iy])) + list(np.unique(MON_L[iy])).index(mon)]["이득 ×1000"] = d[km].mean() * 1000
display(pd.DataFrame(plan).set_index(["연도", "월"]).round(2))
display(pd.DataFrame(rows).set_index("연도").round(4))


예측 초구  학습 초구  그 시즌 학습 초구  이득 ×1000
연도   월                                    
2019 4   2568  12395           0      0.00
     5   2403  14963        2568      1.54
     6   2493  17366        4971      2.31
     7   1791  19859        7464      1.18
     8   2362  21650        9255      3.38
     9   1841  24012       11617      1.62
2020 5   2264  25853           0      0.00
     6   2379  28117        2264      0.26
     7   2233  30496        4643      0.53
     8   1985  32729        6876      1.26
     9   3754  34714        8861     -0.18

,시즌 단위 (model_14),월 단위,이득 ×1000,95% 하한,95% 상한,존 밖 AUC 시즌,존 밖 AUC 월
연도,,,,,,,
2019,2.8835,2.8819,1.6744,0.8179,2.6961,0.5546,0.5558
2020,2.8839,2.8836,0.2900,-0.2997,0.9390,0.5524,0.5534


#### 🔍 0-9 결과: 월 단위 재학습이 두 해 모두 + (2019는 뚜렷, 2020은 작음)
| | 시즌 단위 (model_14) | 월 단위 | 이득 ×1000 (95%) | 존 밖 AUC |
| --- | --- | --- | --- | --- |
| 2019 | 2.8835 | 2.8819 | **+1.67** (+0.82 ~ +2.70) | 0.555 -> 0.556 |
| 2020 | 2.8839 | 2.8836 | +0.29 (−0.30 ~ +0.94) | 0.552 -> 0.553 |

- 시즌 단위 = model_14 재현 확인 (두 해), 첫 달은 같은 모델이라 이득 0
- 달별 이득 ×1000: 2019 +1.5 · +2.3 · +1.2 · +3.4 · +1.6 (모두 +), 2020 +0.3 · +0.5 · +1.3 · −0.2
- 2019 이득은 파라미터를 2019로 고른 게 아님 (model_14 값 그대로) -> 고르는 몫으로 부풀려진 숫자가 아님
- 2020이 작은 이유로 보이는 것: 2019는 학습이 2018 한 해뿐이라 그 시즌 데이터를 더 넣는 몫이 큼 (9월 학습 초구 1.24만 -> 2.40만), 2020은 이미 2018–19 두 해로 학습 (2.59만 -> 3.47만)
  - 앞으로 학습 해가 쌓이면 월 단위의 이득은 더 작아질 것 (구종 model_09도 2019 +0.8 · 2020 +1.0으로 작음)


## 모델

In [11]:
# 0장이 모델 변수 이름을 덮어쓰지 않았는지 대비: 설정 상수를 다시 정함
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]


## 1. 파라미터

- 모두 model_14 값 고정, 바꾸는 것: 위치 상황 보정 재학습 주기 `LOC_RETRAIN = "month"` 하나

In [12]:
# ① 구종 출발점 B4: model_06 값 (고정) - 반감기·alpha·hand_k, 시즌 초 배수 (기준점 = 지난 시즌 전부)
B4_P = dict(half_life=35.4, alpha=7.5, hand_k=5.48, season_f=0.25)
# ① LightGBM 변수: A 이번 타석 상황 + 앞 타석 초구 결과 원-핫 (fc.FIRST_CATS)
FEAT = fc.A_COLS + ["prev_first_" + c for c in fc.FIRST_CATS]
# ② 위치 정규분포: model_13 값 (고정)
LOC_REF, LOC_REF_HL = "decay", 173.0  # 기준점: 분석 대상 투수 구종×손, 전날까지 반감 173구
LOC_HL, K_TYPE, K_HAND = 35.4, 1000.0, 30.0  # 투수 기록 반감기, 투수×구종 당김 k, 투수×구종×손 당김 k
LOC_SEASON_F = 0.707  # 시즌 초 배수
LOC_AC = (None, 12.5, 17.3)  # 모든 카운트 위치 성향 (잔차 기준 반감 None = 누적, δ 반감, δ 당김 k)
LOC_VAR = (200.0, 9.74, False, 316.0)  # 분산 따로 (다른 카운트 퍼짐 반감, k_var, unc, 투수 안 분산 당김 k0)
# ② 위치 상황 보정 (model_14): 24칸 LightGBM, 출발점 = model_13 위치 log 확률
LOC_SIT = ["base1", "base2", "base3", "first_open", "sdiff", "inning", "out_cn", "home", "batorder"]  # 상황 변수 (0-3 묶음 4)
LOC_FEAT = CLASSES + ["same_side"] + LOC_SIT  # 구종 원-핫·타자 손 + 상황
LOC_TRAIN_FIRST = 2018  # 위치 보정 학습 첫 해 (2017은 model_13 위치 없음)
LOC_LGB, LOC_ROUNDS = {"num_leaves": 7, "min_data_in_leaf": 20, "lambda_l2": 1000.0}, 168  # model_14 상황 보정 LightGBM (고정)
LOC_RETRAIN = "month"  # 상황 보정 재학습 주기 (model_16): 그 달 첫 경기 전날까지 데이터로 다시 학습
ZONE_M = 32
# ① LightGBM: model_07 값 (고정)
LGB_P = {"num_leaves": 7, "min_data_in_leaf": 20, "lambda_l2": 1000.0}
LGB_ROUNDS = 730
RETRAIN = "month"  # 재학습 주기: 그 달 첫 경기 전날까지 데이터로 다시 학습 (3월은 4월에, 10월은 9월에 합침)
# 구종 보정 학습 해
TRAIN_FIRST = 2017

## 2. ① 구종: B4 + LightGBM 월 단위 재학습 (model_09과 같음)

### 2-1. B4 (model_06·07·09과 같음)

In [13]:
def b4_online(q, year, half_life, alpha, hand_k, season_f):  # model_06 B4 (n, 9): 지난 시즌 리그 비율 -> 투수 (alpha) -> 투수×손 (hand_k), 시즌 초 배수
    first = year == fp["GYEAR"].min()  # 첫 시즌(2017): 지난 기록 없음 -> 그 시즌 비율 (학습 데이터 출발점용)
    ref = fp.loc[fp["GYEAR"] == year if first else fp["GYEAR"] < year, CLASSES].mean().to_numpy()
    out = []
    for keys in (["pitcher_code"], ["pitcher_code", "same_side"]):
        S, N = fc.lookup(fc.prior_table(fp, keys, CLASSES, half_life, season_f=season_f), q, keys, CLASSES, season_f=season_f)
        out += [S, N[:, :1]]
    S, N, S2, N2 = out
    return fc.shrink(S2, N2, fc.shrink(S, N, ref, alpha), hand_k)

K = len(CLASSES)
P_B4 = np.full((len(fp), K), np.nan)  # 2017–2020
for yr in range(TRAIN_FIRST, fc.TEST_YEAR + 1):
    k = fp["GYEAR"].eq(yr).to_numpy()
    P_B4[k] = b4_online(fp[k], yr, **B4_P)
b4_19 = fc.prob_metrics(P_B4[val], y_type[val])["logloss"]
assert abs(b4_19 - 1.09367) < 1e-5, b4_19  # model_06·07 B4의 2019 log loss
print(f"2019 B4 log loss {b4_19:.5f} = model_06·07 B4 (재현 확인)")
m07 = pd.read_parquet("results/model_07/per_pitch.parquet")  # 첫 달 재현 확인용
m09 = pd.read_parquet("results/model_09/per_pitch.parquet")  # 구종 재현 확인용 (구종은 model_09 그대로)
m13 = pd.read_parquet("results/model_13/per_pitch.parquet")  # 위치 정규분포 재현 확인용
m14 = pd.read_parquet("results/model_14/per_pitch.parquet")  # 비교 대상

2019 B4 log loss 1.09367 = model_06·07 B4 (재현 확인)


### 2-2. 변수 (model_07과 같음)

- A 이번 타석 상황 + 앞 타석 초구 결과, 미래 정보 검사 다시 통과 확인

In [14]:
X_all = pd.concat([fc.situation_features(m, fp), fc.game_flow_features(m, fp)], axis=1)[FEAT]
assert X_all.notna().all().all()
for fn in (fc.situation_features, fc.game_flow_features):
    bad = fc.check_no_future(fn, m, fp[fp["GYEAR"].ge(TRAIN_FIRST)], n=150)
    assert bad.empty, bad
display(X_all.describe().T[["mean", "min", "max"]].round(3))

미래 정보 검사: 150개 중 다른 공 0개
미래 정보 검사: 150개 중 다른 공 0개


,mean,min,max
pc,33.394,0.0,124.0
tto,1.592,1.0,5.0
starter,0.690,0.0,1.0
inning,4.548,1.0,12.0
out_cn,0.970,0.0,2.0
base1,0.359,0.0,1.0
base2,0.194,0.0,1.0
base3,0.100,0.0,1.0
batorder,4.811,1.0,9.0
sdiff,0.369,-21.0,19.0


### 2-2. 월 단위 재학습과 구종 확률 (model_09과 같음)

- 확인: 구종 확률 = model_09 결과 파일 (재현)

In [15]:
y_all = fp["pitch_type"].map(fc.TYPE_IDX).to_numpy()
INIT = np.log(np.clip(P_B4, 1e-6, None))
INIT -= INIT.mean(1, keepdims=True)  # 출발점 (공마다 상수는 softmax에 영향 없음)
BASE = dict(objective="multiclass", num_class=K, learning_rate=0.03, feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1,
            verbose=-1, seed=0, deterministic=True, force_row_wise=True, num_threads=4)  # model_05–07과 같음
yr_all = fp["GYEAR"].to_numpy()
DATE = pd.to_datetime(fp["gameID"].str[:8]).to_numpy()
MON = np.clip(fp["gameID"].str[4:6].astype(int).to_numpy(), 4, 9)  # 재학습 묶음 달: 3월 -> 4월, 10월 -> 9월

def fit_rows(tr):  # 학습 행 tr로 LightGBM (트리 수 고정)
    return lgb.train({**BASE, **LGB_P}, lgb.Dataset(X_all.iloc[tr], y_all[tr], init_score=INIT[tr]), LGB_ROUNDS)

def type_prob(bst, idx):  # B4 출발점 + 트리 보정 -> 구종 확률 (n, 9)
    return np.exp(log_softmax(bst.predict(X_all.iloc[idx], raw_score=True) + INIT[idx], axis=1))

P_TYPE_ALL = np.full((len(fp), K), np.nan)
BST, plan = {}, []
for yr in (fc.VAL_YEAR, fc.TEST_YEAR):
    iy = np.flatnonzero(yr_all == yr)
    for mon in np.unique(MON[iy]):
        te = iy[MON[iy] == mon]
        start = DATE[te].min()  # 그 달 첫 경기일
        tr = np.flatnonzero((yr_all >= TRAIN_FIRST) & (DATE < start))
        assert DATE[tr].max() < start  # 미래 정보 없음
        BST[(yr, mon)] = bst = fit_rows(tr)
        P_TYPE_ALL[te] = type_prob(bst, te)
        plan.append({"연도": yr, "월": mon, "예측 초구": len(te), "학습 초구": len(tr), "그 시즌 학습 초구": int((yr_all[tr] == yr).sum()),
                     "학습 끝": str(pd.Timestamp(DATE[tr].max()).date()), "예측 시작": str(pd.Timestamp(start).date())})
display(pd.DataFrame(plan).set_index(["연도", "월"]))
P11 = {yr: fc.align(m07, fp, yr)[fc.P_TYPE].to_numpy() for yr in (fc.VAL_YEAR, fc.TEST_YEAR)}  # model_07 구종 확률 (fp 그 해 순서)
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:  # 첫 달 = 시즌 단위 (model_07 재현)
    k = MON[msk] == MON[msk].min()
    assert np.allclose(P_TYPE_ALL[msk][k], P11[yr][k], atol=1e-6)
print("첫 달 구종 확률 = model_07 (재현 확인)")
rows = {}
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    for nm, P in [("B4", P_B4[msk]), ("model_07", P11[yr]), (MODEL, P_TYPE_ALL[msk])]:
        r = fc.prob_metrics(P, y_type[msk])
        rows[(yr, nm)] = {"log loss": r["logloss"], "top1": r["top1"], "top2": r["top2"]}
display(pd.DataFrame(rows).T.round(4))
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    assert np.allclose(P_TYPE_ALL[msk], fc.align(m09, fp, yr)[fc.P_TYPE].to_numpy(), atol=1e-6), f"{yr} 구종 확률이 model_09과 다름"
print("구종 확률 = model_09 (재현 확인)")

예측 초구  학습 초구  그 시즌 학습 초구        학습 끝       예측 시작
연도   월                                                  
2019 4   2568  24636           0  2018-10-14  2019-03-23
     5   2403  27204        2568  2019-04-30  2019-05-01
     6   2493  29607        4971  2019-05-31  2019-06-01
     7   1791  32100        7464  2019-06-30  2019-07-02
     8   2362  33891        9255  2019-07-31  2019-08-01
     9   1841  36253       11617  2019-08-31  2019-09-01
2020 5   2264  38094           0  2019-10-01  2020-05-05
     6   2379  40358        2264  2020-05-31  2020-06-02
     7   2233  42737        4643  2020-06-30  2020-07-01
     8   1985  44970        6876  2020-07-31  2020-08-01
     9   3754  46955        8861  2020-08-30  2020-09-01

첫 달 구종 확률 = model_07 (재현 확인)


log loss    top1    top2
2019 B4          1.0937  0.5279  0.8159
     model_07    1.0621  0.5380  0.8174
     model_16    1.0613  0.5385  0.8177
2020 B4          1.0916  0.5264  0.8179
     model_07    1.0570  0.5340  0.8209
     model_16    1.0561  0.5332  0.8218

구종 확률 = model_09 (재현 확인)


## 3. ② 위치: model_13 정규분포 + 상황 보정 (월 단위 재학습)

### 3-1. model_13 위치 (그대로)

In [17]:
# 24칸 적분 함수: fc.zone_from_gauss_cov(g, q, m=32) (model_03부터)
def zone_probs(g, q, cov=True):  # 24칸 확률: cov=True 공분산 반영 (model_03) / False 독립 가정 (model_02까지)
    return fc.zone_from_gauss_cov(g, q, ZONE_M) if cov else fc.zone_from_gauss(g, q)

L_COLS = MOM + ZONES  # 당길 열: 위치 모멘트 5개(x, 높이 비율, 제곱, 곱) + 24칸 원-핫
K1, K2 = ["pitcher_code", "pitch_type"], ["pitcher_code", "pitch_type", "same_side"]  # 투수×구종 / 투수×구종×손
KR = ["pitch_type", "same_side"]  # 기준점 그룹: 분석 대상 투수 구종×손

def loc_league(q, year, ref="all", ref_hl=None):  # 기준점 (n, 29): "all" 지난 시즌 전부 / "last" 직전 시즌만 / "decay" 전날까지 반감기 ref_hl(공 수) 가중
    past = fp[fp["GYEAR"] < year]  # 지난 시즌까지
    overall = past[L_COLS].mean().to_numpy()  # 그 구종×손 기록이 없을 때 쓰는 전체 평균 (지난 시즌까지)
    if ref == "decay":
        S, N = fc.lookup(fc.prior_table(fp, KR, L_COLS, ref_hl), q, KR, L_COLS)  # 구종×손별 전날까지 가중 합계, 개수
        return np.where(N > 0, S / np.maximum(N, 1e-12), overall)  # 가중 평균, 기록 없으면 전체
    if ref == "last":
        past = fp[fp["GYEAR"] == year - 1]  # 직전 시즌만
    lg = past.groupby(KR)[L_COLS].mean()
    pr = lg.reindex(pd.MultiIndex.from_frame(q[KR]))  # 공마다 그 구종×손 행
    return pr.fillna(pd.Series(overall, index=L_COLS)).to_numpy()

def loc_sn(q, half_life, season_f, cache=True):  # 투수×구종, 투수×구종×손 전날까지 초구 위치 합계 S·개수 N 두 쌍 (시즌 초 배수 반영)
    return [fc.lookup(fc.prior_table(fp, keys, L_COLS, half_life, season_f=season_f, cache=cache), q, keys, L_COLS, season_f=season_f)
            for keys in (K1, K2)]

def shift_mean(M, dx, dz):  # 모멘트 M (n, 29)의 평균만 (dx, dz) 옮김, 분산·공분산 그대로 (24칸 원-핫 열은 정규분포에 안 쓰여 그대로)
    M = M.copy()
    mx, mz = M[:, 0], M[:, 1]
    nx, nz = mx + dx, mz + dz
    M[:, 2] += nx ** 2 - mx ** 2  # E[x²] = 분산 + 평균²
    M[:, 3] += nz ** 2 - mz ** 2
    M[:, 4] += nx * nz - mx * mz  # E[x·z] = 공분산 + 평균 곱
    M[:, 0], M[:, 1] = nx, nz
    return M

# 0-0 아닌 공 (구종 의심 제외): 모든 카운트 위치 성향 재료
MO = m[~m["label_doubt"] & ~(m["ball_before"].eq(0) & m["strike_before"].eq(0))].copy()
MO["x"], MO["zn"] = MO["plate_x_in_CM"], MO["z_norm"]  # 좌우 cm (타자 기준), 높이 비율 (존 아랫선 0, 윗선 1)
MO["cnt"] = MO["ball_before"].astype(int).astype(str) + "-" + MO["strike_before"].astype(int).astype(str)  # 카운트
CK = ["cnt", "pitch_type", "same_side"]  # 카운트 효과 그룹
_MO_RES = {}

def mo_resid(ref_hl):  # 0-0 아닌 공마다 위치 − 분석 대상 투수 그 카운트×구종×손 전날까지 평균 (반감 ref_hl구) -> 잔차 표 (rx 좌우 cm, rz 높이 비율)
    if ref_hl not in _MO_RES:
        S, N = fc.lookup(fc.prior_table(MO, CK, ["x", "zn"], ref_hl, cache=False), MO, CK, ["x", "zn"])
        ref = np.where(N > 0, S / np.maximum(N, 1e-12), np.nan)  # 그 카운트×구종×손 기록이 없으면 빈 값 (셈에서 빠짐)
        _MO_RES[ref_hl] = MO[K2 + ["game_date", "gameID", "pit_no"]].assign(rx=MO["x"].to_numpy() - ref[:, 0], rz=MO["zn"].to_numpy() - ref[:, 1])
    return _MO_RES[ref_hl]

def ac_sn(q, ref_hl, hl_b, season_f, cache=True):  # 공마다 투수×구종×손 다른 카운트 잔차 전날까지 합 S·개수 N (n, 2)
    d = mo_resid(ref_hl)
    return fc.lookup(fc.prior_table(d, K2, ["rx", "rz"], hl_b, season_f=season_f, cache=cache), q, K2, ["rx", "rz"], season_f=season_f)

MCOLS = ["x", "zn", "xx", "zz", "xz"]  # 위치 모멘트 (좌우, 높이 비율, 제곱, 곱)
_MO_MOM, _WT = {}, {}

def mo_mom(ref_hl):  # 다른 카운트 잔차 모멘트 표 (x = 좌우 잔차, zn = 높이 비율 잔차, 제곱·곱)
    if ref_hl not in _MO_MOM:
        d = mo_resid(ref_hl)
        _MO_MOM[ref_hl] = d.assign(xx=d["rx"] ** 2, zz=d["rz"] ** 2, xz=d["rx"] * d["rz"]).rename(columns={"rx": "x", "rz": "zn"})
    return _MO_MOM[ref_hl]

def within_tab(name, data, year):  # 구종×손별 투수 안 분산·공분산 (지난 시즌까지 투수×구종×손 분산의 (공 수 − 1) 가중 평균) 표와 전체 값
    if (name, year) not in _WT:
        past = data[data["game_date"].dt.year < year].dropna(subset=["x", "zn"])
        g = past.groupby(K2)
        n = g.size()
        w = (n - 1).clip(lower=0)
        c = pd.DataFrame({"vx": g["x"].var(), "vz": g["zn"].var(),
                          "cxz": (g["xz"].mean() - g["x"].mean() * g["zn"].mean()) * n / (n - 1).clip(lower=1)}).fillna(0)  # 표본 분산·공분산
        num = c.mul(w, axis=0).groupby(level=["pitch_type", "same_side"]).sum()
        wg = w.groupby(level=["pitch_type", "same_side"]).sum()  # 구종×손별 자유도 합 (표본 크기)
        _WT[(name, year)] = (num.div(wg, axis=0), c.mul(w, axis=0).sum() / w.sum(), wg)
    return _WT[(name, year)]

def v0_of(q, name, data, year, k0=0.0):  # 공마다 그 구종×손 투수 안 분산·공분산 (n, 3): 자유도 합이 적으면 전체 구종 값 쪽으로 k0만큼 당김
    tab, allv, wg = within_tab(name, data, year)
    idx = pd.MultiIndex.from_frame(q[KR])
    a = allv.to_numpy()[None, :]  # 전체 구종 투수 안 분산
    v = tab.reindex(idx).to_numpy()
    n = wg.reindex(idx).fillna(0).to_numpy()[:, None]  # 그 구종×손 자유도 합
    v = np.where(np.isnan(v), a, v)
    return np.where(n + k0 > 0, (n * v + k0 * a) / np.maximum(n + k0, 1e-12), a)  # 예: 너클 초구 3구(자유도 2)면 거의 전체 값

def own_var(q, year, ref_hl, hl_var, season_f, k0=0.0, cache=True):  # 공마다 다른 카운트 퍼짐 -> 초구 단위로 바꾼 자기 분산·공분산 (n, 3), (가중) 공 수 (n,), V0 (n, 3)
    data = mo_mom(ref_hl)
    S, N = fc.lookup(fc.prior_table(data, K2, MCOLS, hl_var, season_f=season_f, cache=cache), q, K2, MCOLS, season_f=season_f)
    e = S / np.maximum(N, 1e-12)
    v = np.column_stack([e[:, 2] - e[:, 0] ** 2, e[:, 3] - e[:, 1] ** 2, e[:, 4] - e[:, 0] * e[:, 1]])  # 자기 잔차 분산·공분산
    V0 = v0_of(q, "초구", fp, year, k0)
    V0m = v0_of(q, ("다른 카운트", ref_hl), data, year, k0)
    r = np.sqrt(np.maximum(v[:, :2], 0) / V0m[:, :2])  # 좌우·높이 SD 비율 (자기 / 투수 안)
    v = np.column_stack([V0[:, 0] * r[:, 0] ** 2, V0[:, 1] * r[:, 1] ** 2, v[:, 2] / np.sqrt(V0m[:, 0] * V0m[:, 1]) * np.sqrt(V0[:, 0] * V0[:, 1])])
    n = N[:, 0]
    return np.where(n[:, None] > 0, np.nan_to_num(v), 0.0), n, V0

def apply_var(M, v, n, V0, k_var, unc, N2, k_hand):  # 모멘트 M의 분산·공분산을 따로 당긴 값으로 바꿈 (평균 그대로)
    V = (n[:, None] * v + k_var * V0) / (n[:, None] + k_var)  # 투수 안 분산 쪽으로 k_var만큼
    if unc:
        V = V * (1 + 1 / (N2[:, :1] + k_hand))  # 평균을 덜 알수록 넓힘
    M = M.copy()
    mx, mz = M[:, 0], M[:, 1]
    M[:, 2], M[:, 3], M[:, 4] = V[:, 0] + mx ** 2, V[:, 1] + mz ** 2, V[:, 2] + mx * mz  # 분산 -> 모멘트
    return M

def loc_online(q, year, half_life, k_type, k_hand, ref="all", ref_hl=None, season_f=1.0, ac=None, var=None):  # 위치 모멘트 (n, 29)
    P = loc_league(q, year, ref, ref_hl)  # 출발: 기준점
    if ac is not None:  # 모든 카운트 위치 성향 (model_12)
        S, N = ac_sn(q, ac[0], ac[1], season_f)
        d = S / (N + ac[2])
        P = shift_mean(P, d[:, 0], d[:, 1])
    (S1, N1), (S2, N2) = loc_sn(q, half_life, season_f)
    M = fc.shrink(S2, N2, fc.shrink(S1, N1, P, k_type), k_hand)  # 투수×구종 -> 투수×구종×손
    if var is not None:  # 분산 따로 (model_13): var = (hl_var, k_var, unc, k0)
        v, n, V0 = own_var(q, year, None if ac is None else ac[0], var[0], season_f, var[3])
        M = apply_var(M, v, n, V0, var[1], var[2], N2, k_hand)
    return M

def loc_given_type(q, year, t, half_life, k_type, k_hand, ref="all", ref_hl=None, season_f=1.0, ac=None, var=None, cov=True):  # 구종이 t일 때 24칸 확률 (n, 24)과 좌표 정규분포
    qt = fc.as_type(q, t)
    g = fc.gauss(loc_online(qt, year, half_life, k_type, k_hand, ref, ref_hl, season_f, ac, var), qt)
    return zone_probs(g, qt, cov), g


LOC17 = dict(half_life=LOC_HL, k_type=K_TYPE, k_hand=K_HAND, ref=LOC_REF, ref_hl=LOC_REF_HL, season_f=LOC_SEASON_F, ac=LOC_AC, var=LOC_VAR)
PZ_TRUE = np.full((len(fp), len(ZONES)), np.nan)  # 실제 구종 기준 model_13 24칸 확률 (2018–2020)
for yr in range(LOC_TRAIN_FIRST, fc.TEST_YEAR + 1):
    k = fp["GYEAR"].eq(yr).to_numpy()
    PZ_TRUE[k] = loc_given_type(fp[k], yr, fp.loc[k, "pitch_type"].to_numpy(), **LOC17)[0]
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    ll_ = -np.log(np.clip(PZ_TRUE[msk][np.arange(msk.sum()), y_zone[msk]], 1e-12, None))
    assert np.allclose(ll_, fc.align(m13, fp, yr)["ll_zone_true"].to_numpy()), f"{yr} model_13 위치 재현 실패"
print("실제 구종 기준 24칸 = model_13 (재현 확인), 2018 계산된 공:", int(np.isfinite(PZ_TRUE[fp["GYEAR"].eq(2018).to_numpy()]).all(1).sum()))

실제 구종 기준 24칸 = model_13 (재현 확인), 2018 계산된 공: 12395


### 3-2. 월 단위 재학습

- 변수 표는 model_14과 같음 (상황 변수 미래 정보 검사 다시 통과 확인)
- 달마다: 학습 = 그 달 첫 경기일보다 앞선 2018년부터의 모든 초구 (`DATE < 시작일`) -> 그 달 예측, 트리 168 고정
- 확인: 첫 달 = 시즌 단위 (model_14 재현), 학습 데이터 끝 < 예측 달 시작 (미래 정보 없음)

In [18]:
from scipy.special import log_softmax
for fn in (fc.situation_features, fc.opponent_features):
    bad = fc.check_no_future(fn, m, fp[fp["GYEAR"].ge(LOC_TRAIN_FIRST)], n=150)
    assert bad.empty, bad
SIT_L = fc.situation_features(m, fp)
XLOC = pd.concat([fp[CLASSES + ["same_side"]], SIT_L, fc.opponent_features(m, fp)[["home"]]], axis=1)
XLOC["first_open"] = (((SIT_L["base2"] + SIT_L["base3"]) > 0) & SIT_L["base1"].eq(0)).astype(int)  # 득점권인데 1루 비었음
XLOC = XLOC[LOC_FEAT]
NZ = len(ZONES)
INIT_L = np.log(np.clip(PZ_TRUE, 1e-6, None))
INIT_L -= INIT_L.mean(1, keepdims=True)  # 출발점
yr_l = fp["GYEAR"].to_numpy()
BASE_L = dict(objective="multiclass", num_class=NZ, learning_rate=0.05, feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1,
              verbose=-1, seed=0, deterministic=True, force_row_wise=True, num_threads=4)


DATE_L = fp["game_date"].to_numpy()
MON_L = np.clip(fp["game_date"].dt.month.to_numpy(), 4, 9)  # 재학습 묶음 달: 3월 -> 4월, 10월 -> 9월

def fit_rows(tr):  # 학습 행 tr로 상황 보정 LightGBM (model_14 파라미터, 트리 168)
    return lgb.train({**BASE_L, **LOC_LGB}, lgb.Dataset(XLOC.iloc[tr], y_zone[tr], init_score=INIT_L[tr]), LOC_ROUNDS)

BST_L, plan = {}, []  # (연도, 달) -> 그 달 예측용 모델
for yr in (fc.VAL_YEAR, fc.TEST_YEAR):
    iy = np.flatnonzero(yr_l == yr)
    for mon in np.unique(MON_L[iy]):
        te = iy[MON_L[iy] == mon]
        start = DATE_L[te].min()  # 그 달 첫 경기일
        tr = np.flatnonzero((yr_l >= LOC_TRAIN_FIRST) & (DATE_L < start))
        assert DATE_L[tr].max() < start  # 미래 정보 없음
        BST_L[(yr, mon)] = fit_rows(tr)
        plan.append({"연도": yr, "월": mon, "예측 초구": len(te), "학습 초구": len(tr), "그 시즌 학습 초구": int((yr_l[tr] == yr).sum()),
                     "학습 끝": str(pd.Timestamp(DATE_L[tr].max()).date()), "예측 시작": str(pd.Timestamp(start).date())})
display(pd.DataFrame(plan).set_index(["연도", "월"]))

def loc_correct(P, q, year, t=None):  # model_13 24칸 확률 P (n, 24) -> 상황 보정 확률 (공마다 그 달 모델), t = 가정한 구종 (None이면 q의 실제 구종)
    X = XLOC.loc[q.index].copy()
    if t is not None:
        X.loc[:, CLASSES] = np.tile((np.array(CLASSES) == t).astype(float), (len(X), 1))  # 구종 원-핫을 t로
    init = np.log(np.clip(P, 1e-6, None))
    init -= init.mean(1, keepdims=True)
    mon = MON_L[fp.index.get_indexer(q.index)]
    out = np.empty_like(P)
    for mo in np.unique(mon):
        k = mon == mo
        out[k] = np.exp(log_softmax(BST_L[(year, mo)].predict(X[k], raw_score=True) + init[k], axis=1))
    return out

for yr, q, msk in [(fc.VAL_YEAR, q_val, val), (fc.TEST_YEAR, q_test, test)]:  # 첫 달 = model_14 (시즌 단위) 재현
    P = loc_correct(PZ_TRUE[msk], q, yr)
    ll = -np.log(np.clip(P[np.arange(msk.sum()), y_zone[msk]], 1e-12, None))
    first = MON_L[msk] == MON_L[msk].min()
    assert np.allclose(ll[first], fc.align(m14, fp, yr)["ll_zone_true"].to_numpy()[first], atol=1e-6), f"{yr} 첫 달이 model_14과 다름"
print("첫 달 = model_14 (재현 확인)")

미래 정보 검사: 150개 중 다른 공 0개
미래 정보 검사: 150개 중 다른 공 0개


예측 초구  학습 초구  그 시즌 학습 초구        학습 끝       예측 시작
연도   월                                                  
2019 4   2568  12395           0  2018-10-14  2019-03-23
     5   2403  14963        2568  2019-04-30  2019-05-01
     6   2493  17366        4971  2019-05-31  2019-06-01
     7   1791  19859        7464  2019-06-30  2019-07-02
     8   2362  21650        9255  2019-07-31  2019-08-01
     9   1841  24012       11617  2019-08-31  2019-09-01
2020 5   2264  25853           0  2019-10-01  2020-05-05
     6   2379  28117        2264  2020-05-31  2020-06-02
     7   2233  30496        4643  2020-06-30  2020-07-01
     8   1985  32729        6876  2020-07-31  2020-08-01
     9   3754  34714        8861  2020-08-30  2020-09-01

첫 달 = model_14 (재현 확인)


### 3-3. 진단 (실제 구종 기준)

- 달별 model_14 대비 이득 (0-9와 같아야 함), 주자 상황·아웃별 존 밖(C) 실제 − 예측, 존 밖 AUC

In [19]:
def runner3(q):
    return np.select([(q["base2"] + q["base3"]) > 0, q["base1"] > 0], ["득점권", "1루만"], "없음")

rows = []
for yr, q, msk in [(fc.VAL_YEAR, q_val, val), (fc.TEST_YEAR, q_test, test)]:
    i, yz = np.arange(len(q)), y_zone[msk]
    P20 = loc_correct(PZ_TRUE[msk], q, yr)
    a18 = fc.align(m14, fp, yr)
    ll18, ll20 = a18["ll_zone_true"].to_numpy(), -np.log(np.clip(P20[i, yz], 1e-12, None))
    oc = (fc.ZONE_GROUP[yz] == "C").astype(float)
    d = pd.DataFrame({"월": MON_L[msk], "주자": runner3(SIT_L.loc[q.index]), "아웃": SIT_L.loc[q.index, "out_cn"].clip(upper=2).to_numpy(),
                      "C 실제": oc, "C 20": P20[:, fc.ZONE_GROUP == "C"].sum(1), "이득": (ll18 - ll20) * 1000})
    for var in ["월", "주자", "아웃"]:
        for v, g in d.groupby(var):
            rows.append({"연도": yr, "변수": var, "값": str(v), "공": len(g), "이득 ×1000 (model_14 대비)": g["이득"].mean(),
                         "rC model_16": (g["C 실제"] - g["C 20"]).mean()})
    rows.append({"연도": yr, "변수": "전체", "값": "", "공": len(d), "이득 ×1000 (model_14 대비)": d["이득"].mean(),
                 "rC model_16": (d["C 실제"] - d["C 20"]).mean(), "존 밖 AUC model_16": auc(d["C 20"].to_numpy(), oc)})
display(pd.DataFrame(rows).set_index(["변수", "값", "연도"]).unstack("연도").round(4))

공          이득 ×1000 (model_14 대비)         rC model_16         존 밖 AUC model_16        
연도         2019     2020                   2019    2020        2019    2020             2019    2020
변수 값                                                                                                
아웃 0     4645.0   4375.0                 2.9710 -0.1456     -0.0154 -0.0044              NaN     NaN
   1     4500.0   4222.0                 1.4865  0.1876     -0.0010 -0.0046              NaN     NaN
   2     4313.0   4018.0                 0.4741  0.8721     -0.0109  0.0176              NaN     NaN
월  4     2568.0      NaN                 0.0000     NaN     -0.0144     NaN              NaN     NaN
   5     2403.0   2264.0                 1.5401  0.0000     -0.0050 -0.0074              NaN     NaN
   6     2493.0   2379.0                 2.3136  0.2625     -0.0124  0.0143              NaN     NaN
   7     1791.0   2233.0                 1.1795  0.5342      0.0187  0.0001              NaN     NaN
   8     2362.0   1985.0                 3.3772  1.2618     -0.0175 -0.0030              NaN     NaN
   9     1841.0   3754.0                 1.6166 -0.1766     -0.0191  0.0055              NaN     NaN
전체      13458.0  12615.0                 1.6744  0.2900     -0.0091  0.0025           0.5558  0.5534
주자 1루만   2886.0   2723.0                 0.5843  1.5444     -0.0259  0.0037              NaN     NaN
   득점권   3346.0   3136.0                 1.3853  0.4182      0.0277  0.0153              NaN     NaN
   없음    7226.0   6756.0                 2.2437 -0.2750     -0.0195 -0.0039              NaN     NaN

#### 🔍 3-3 결과: 0-9와 같음, 2019는 고르게 좋아지고 2020은 작음
- 전체 이득 ×1000 (model_14 대비, 실제 구종): 2019 +1.67, 2020 +0.29 (0-9와 같음)
- 달별: 2019 +1.5 · +2.3 · +1.2 · +3.4 · +1.6 (모두 +), 2020 +0.3 · +0.5 · +1.3 · −0.2 (첫 달은 같은 모델이라 0)
- 주자 상황·아웃별: 2019는 모두 +, 2020은 1루만 +1.5 · 2아웃 +0.9 말고는 0 근처
- 존 밖 AUC: 2019 0.555 -> 0.556, 2020 0.552 -> 0.553


## 4. 실전 예측 (예측 구종으로 위치 이어서)

- model_12과 같음, 위치만 3-1에서 고른 값 (분산 따로 포함)

In [20]:
def predict(q, year, Pt):  # 초구 q의 공별 예측·점수
    n, i = len(q), np.arange(len(q))
    yt, yz = q["pitch_type"].map(fc.TYPE_IDX).to_numpy(), q["zone"].map(fc.ZONE_IDX).to_numpy()  # 실제 구종·칸 번호
    nll = lambda p: -np.log(np.clip(p, 1e-12, None))  # 공별 log loss (prob_metrics와 같은 방식으로 자름)

    # ① 구종 확률 (n, 9): 2장 결과를 받음

    # ② 위치: 구종 9개 각각 가정 -> 24칸 확률, 실제 위치의 로그 밀도, 예측 중심 (좌우, 높이)
    PZ, LD, MX, MZ = [], [], [], []
    for t in CLASSES:
        P, g = loc_given_type(q, year, t, LOC_HL, K_TYPE, K_HAND, LOC_REF, LOC_REF_HL, LOC_SEASON_F, LOC_AC, LOC_VAR)
        P = loc_correct(P, q, year, t)  # 상황 보정 (model_14), 정규분포 g는 model_13 그대로
        PZ.append(P); LD.append(fc.log_dens(g, q)); MX.append(g[0]); MZ.append(g[1])
    PZ, LD, MX, MZ = np.stack(PZ, 1), np.stack(LD, 1), np.stack(MX, 1), np.stack(MZ, 1)  # (n, 구종, ...)
    joint = Pt[:, :, None] * PZ  # 구종 확률 × 그 구종일 때 칸 확률 = 구종×칸 결합 확률 (n, 9, 24)
    P_zone = joint.sum(1)  # 실전 24칸 확률 = 구종으로 합침
    # 좌표: 구종 확률로 섞은 정규분포에서 실제 위치의 로그 밀도
    ld_mix = np.log(np.einsum("nt,nt->n", Pt, np.exp(LD - LD.max(1, keepdims=True)))) + LD.max(1)  #  공마다 최댓값 빼기 -> exp -> 섞기 -> log (계산 안전)
    dist_true = np.hypot(q["x"] - MX[i, yt], q["CROSS_PLATE_Z_CM"] - MZ[i, yt])  # 실제 구종 중심과의 거리 cm
    dist_mix = np.hypot(q["x"] - (Pt * MX).sum(1), q["CROSS_PLATE_Z_CM"] - (Pt * MZ).sum(1))  # 구종 확률로 가중한 중심과의 거리 cm

    t_hat = np.array(CLASSES)[Pt.argmax(1)]  # 1등 구종 이름

    # 공별 결과 표 (필수: 키, 구종 확률 9개, 실전 24칸 확률 / 나머지는 선택 열)
    return pd.concat([
        q[["gameID", "pit_no", "pitcher_code", "GYEAR", "pitch_type", "zone"]].reset_index(drop=True),
        pd.DataFrame(Pt, columns=fc.P_TYPE),  # p_type_<구종>
        pd.DataFrame(P_zone, columns=fc.P_ZONE),  # p_zone_<칸>
        pd.DataFrame({"t_hat": t_hat}),  # 1등 구종 이름
        pd.DataFrame({
            "ll_type": nll(Pt[i, yt]),  # 구종 log loss
            "ll_zone": nll(P_zone[i, yz]),  # 24칸 log loss (실전)
            "ll_zone_true": nll(PZ[i, yt, yz]),  # 24칸 log loss (실제 구종)
            "ll_joint": nll(joint[i, yt, yz]),  # 구종×칸 log loss
            "nll_coord": -ld_mix,  # 좌표 NLL (실전)
            "nll_coord_true": -LD[i, yt],  # 좌표 NLL (실제 구종)
            "dist": np.asarray(dist_mix),  # 좌표 거리 cm (실전)
            "dist_true": np.asarray(dist_true),  # 좌표 거리 cm (실제 구종)
        })], axis=1)

In [21]:
# 2019 검증 + 2020 평가 초구 전부 예측
pp = pd.concat([predict(q_val, fc.VAL_YEAR, P_TYPE_ALL[val]), predict(q_test, fc.TEST_YEAR, P_TYPE_ALL[test])], ignore_index=True)
print(pp.shape)
# 확인: 구종·좌표(정규분포)는 model_14과 같아야 함, 24칸만 바뀜
chk = pp.merge(m14, on=fc.KEYS, suffixes=("", "_14"))
same = ["ll_type", "nll_coord_true", "dist_true"] + fc.P_TYPE
bad = [c for c in same if not np.allclose(chk[c], chk[c + "_14"], equal_nan=True, atol=1e-6)]
assert len(chk) == len(pp) and not bad, f"model_14과 다른 열: {bad}"
print("구종·좌표 = model_14 (재현 확인), 24칸 확률만 바뀜")

(26073, 48)
구종·좌표 = model_14 (재현 확인), 24칸 확률만 바뀜


## 5. 평가

- 채점은 `fc.evaluate`, model_14과 나란히 / 경기 단위 부트스트랩 (model_16 − model_14, 2020)
- 2020은 확인용, 너클 뺀 값도 봄 (4-27)

In [22]:
def eval_years(pp_):  # 2019·2020 주요 지표 표
    return pd.DataFrame({"검증 2019": fc.evaluate(pp_, fp, fc.VAL_YEAR), "평가 2020": fc.evaluate(pp_, fp, fc.TEST_YEAR)})

summary = eval_years(pp)
m14_sum = eval_years(m14)
display(pd.concat({MODEL: summary, "model_14": m14_sum, "차이 (model_14 대비)": summary - m14_sum}, axis=1).round(4))

model_16          model_14          차이 (model_14 대비)        
                     검증 2019  평가 2020  검증 2019  평가 2020          검증 2019 평가 2020
구종 logloss            1.0613   1.0561   1.0613   1.0561           0.0000  0.0000
구종 top1               0.5385   0.5332   0.5385   0.5332           0.0000  0.0000
구종 top2               0.8177   0.8218   0.8177   0.8218           0.0000  0.0000
구종 top3               0.9374   0.9412   0.9374   0.9412           0.0000  0.0000
구종 macro F1           0.3226   0.3545   0.3226   0.3545           0.0000  0.0000
24칸 logloss (실전)      2.9413   2.9402   2.9429   2.9403          -0.0016 -0.0001
24칸 top1              0.1329   0.1331   0.1329   0.1328           0.0000  0.0003
24칸 top3              0.2992   0.2999   0.2951   0.2978           0.0041  0.0021
H/S/C logloss         1.0733   1.0742   1.0736   1.0744          -0.0003 -0.0001
H/S/C 정확도             0.4010   0.3981   0.4014   0.3983          -0.0004 -0.0002
24칸 logloss (실제 구종)   2.8819   2.8836   2.8835   2.8839          -0.0017 -0.0003
구종×칸 logloss          3.9432   3.9397   3.9448   3.9399          -0.0017 -0.0003
좌표 평균거리 cm           32.6377  31.8198  32.6377  31.8198           0.0000  0.0000
좌표 NLL                9.3124   9.2662   9.3124   9.2662           0.0000  0.0000

In [23]:
display(fc.bootstrap_diff(fc.pitch_scores(m14, fp), fc.pitch_scores(pp, fp)).round(4))  # 음수 = 이 모델이 좋음 (b − a)
for yr in (fc.VAL_YEAR, fc.TEST_YEAR):
    a18, a20 = fc.align(m14, fp, yr), fc.align(pp, fp, yr)
    nk = a18["pitch_type"].ne("KNUC").to_numpy()
    d_true = a20["ll_zone_true"].to_numpy() - a18["ll_zone_true"].to_numpy()
    print(f"{yr} 24칸 실제 구종 차이 (model_16 − model_14): 전체 {d_true.mean():+.5f}, 너클 뺌 {d_true[nk].mean():+.5f}")

,차이 (b−a),95% 하한,95% 상한,판정
ll_type,0.0000,0.0000,0.0000,차이 불확실
ll_zone,-0.0001,-0.0008,0.0004,차이 불확실
ll_joint,-0.0003,-0.0009,0.0003,차이 불확실
nll_coord,0.0000,0.0000,0.0000,차이 불확실
dist,0.0000,0.0000,0.0000,차이 불확실


2019 24칸 실제 구종 차이 (model_16 − model_14): 전체 -0.00167, 너클 뺌 -0.00167
2020 24칸 실제 구종 차이 (model_16 − model_14): 전체 -0.00029, 너클 뺌 -0.00031


### 🥇1등🥇
#### 2019 24칸 실제 구종 −0.0017 / 실전 −0.0016, 2020 실제 구종 −0.0003 / 실전 −0.0001, 나빠진 점수 없음 -> 채택
- 재학습 주기만 바꿈 (변수·파라미터·트리 수·위치 정규분포·구종은 model_14 그대로), 첫 달 = model_14 재현 확인
- 부트스트랩 (2020, model_16 − model_14): 6개 점수 모두 차이 불확실, 나빠짐 없음
- 2019 숫자는 파라미터를 2019로 고르지 않아 부풀려지지 않음, 2020이 작은 이유는 학습 데이터 양 (2019는 2018 한 해뿐 -> 그 시즌 데이터를 더 넣는 몫이 큼)
- **2020 실전 효과는 사실상 0 (−0.0001)**, 학습 해가 쌓일수록 이득은 더 작아질 것
- 운영: 구종(model_09)과 같이 매달 첫 경기 전에 그때까지 데이터로 다시 학습
- **채택 -> 다음 비교 출발점** (사용자 결정 2026-11-11)

## 6. 저장

- `results/model_16/` (채택하면 `compare.ipynb` `ADOPTED`에 추가)

In [24]:
fc.save_results(MODEL, pp, {"summary": summary})

저장: results\model_16 ['per_pitch.parquet', 'summary.csv']
